# 01 — Importación, exploración y normalización de datos

Importamos y exploramos los archivos de `data/raw`. La normalización se realiza en copias con columnas adicionales; los archivos fuente y las columnas originales se conservan.

## Descripción de los datos

Los datos son sintéticos y contienen inconsistencias deliberadas.

| Archivo | Qué representa cada registro | Contenido principal |
| --- | --- | --- |
| `leads.csv` | Un registro de un potencial cliente recibido por un canal | Contacto, empresa, punto de venta, modelo de interés y estado de gestión. |
| `catalogo_motos.csv` | Una referencia de moto | Marca, línea, cilindraje, segmento, precio y disponibilidad. Los puntos disponibles vienen separados por `&#124;`; las unidades no están desglosadas por punto. |
| `asesores.csv` | Un asesor comercial | Empresa, punto de venta, capacidad diaria, estado activo y fecha de ingreso. |
| `historico_cierres.csv` | Un lead histórico con su desenlace registrado | Características de la gestión y resultado: Cerrado, Perdido o Sin gestión. |
| `conversaciones.json` | Una conversación de WhatsApp asociada a un `lead_id` | Fecha de inicio y lista de mensajes con emisor, hora y texto. |

Los identificadores permiten relacionar archivos, pero todavía no hemos validado su unicidad ni sus correspondencias. Las fechas permanecen como texto y los teléfonos se cargan como cadenas.


In [ ]:
import json
from pathlib import Path

import pandas as pd

In [ ]:
# Permite ejecutar desde la raíz del proyecto o desde notebooks/.
raiz_proyecto = Path.cwd()
if not (raiz_proyecto / 'data' / 'raw').is_dir():
    raiz_proyecto = raiz_proyecto.parent

ruta_datos = raiz_proyecto / 'data' / 'raw'
if not ruta_datos.is_dir():
    raise FileNotFoundError('Ejecuta el notebook desde la raíz del proyecto o desde notebooks/.')

ruta_datos

In [ ]:
leads = pd.read_csv(ruta_datos / 'leads.csv', encoding='utf-8', dtype={'telefono': 'string'})
catalogo_motos = pd.read_csv(ruta_datos / 'catalogo_motos.csv', encoding='utf-8')
asesores = pd.read_csv(ruta_datos / 'asesores.csv', encoding='utf-8')
historico_cierres = pd.read_csv(ruta_datos / 'historico_cierres.csv', encoding='utf-8')

In [ ]:
# Conservamos la estructura del JSON, incluida la lista de mensajes de cada conversación.
with (ruta_datos / 'conversaciones.json').open(encoding='utf-8') as archivo:
    conversaciones = json.load(archivo)

In [ ]:
# Comprobación básica de la carga.
pd.DataFrame({
    'archivo': ['leads.csv', 'catalogo_motos.csv', 'asesores.csv', 'historico_cierres.csv', 'conversaciones.json'],
    'registros': [len(leads), len(catalogo_motos), len(asesores), len(historico_cierres), len(conversaciones)],
})

## 1. Primeras filas

Revisamos cinco registros de cada CSV para reconocer las columnas y sus valores originales.

In [ ]:
leads.head()

In [ ]:
catalogo_motos.head()

In [ ]:
asesores.head()

In [ ]:
historico_cierres.head()

## 2. Estructura y tipos de datos

`info()` muestra el número de registros, las columnas, los valores no nulos y los tipos que pandas asignó al importar.

In [ ]:
tablas = {
    'leads': leads,
    'catalogo_motos': catalogo_motos,
    'asesores': asesores,
    'historico_cierres': historico_cierres,
}

for nombre, tabla in tablas.items():
    print(f'\n{nombre}: {tabla.shape[0]} filas y {tabla.shape[1]} columnas')
    tabla.info()


## 3. Valores faltantes

Mostramos cantidad y porcentaje de nulos por columna. El porcentaje se calcula sobre las filas de cada tabla. `isna()` identifica los nulos reconocidos por pandas; textos con espacios o categorías como `NO_INFORMA` no se cuentan como nulos. Un valor faltante no necesariamente representa un error: su significado depende del campo.


In [ ]:
from IPython.display import display

for nombre, tabla in tablas.items():
    print(nombre)
    resumen_nulos = pd.DataFrame({
        'nulos': tabla.isna().sum(),
        'porcentaje_nulos': tabla.isna().mean().mul(100).round(2),
    })
    display(resumen_nulos.sort_values('nulos', ascending=False))


## 4. Valores distintos en campos categóricos

Revisamos las categorías y sus frecuencias sin unificar mayúsculas, tildes ni espacios. `dropna=False` incluye los valores nulos en los conteos.


In [ ]:
columnas_categoricas = {
    'leads': ['canal', 'empresa_id', 'punto_venta_id', 'estado_gestion', 'ciudad'],
    'catalogo_motos': ['marca', 'segmento'],
    'asesores': ['empresa_id', 'punto_venta_id', 'activo'],
    'historico_cierres': [
        'canal', 'empresa_id', 'punto_venta_id', 'manifesto_cuota_inicial',
        'forma_pago_declarada', 'pidio_cita', 'desenlace',
    ],
}

for nombre, columnas in columnas_categoricas.items():
    for columna in columnas:
        print(f'{nombre} — {columna}')
        display(tablas[nombre][columna].value_counts(dropna=False).rename('registros').to_frame())


## 5. Estructura de las conversaciones

El JSON es una lista de conversaciones. Inspeccionamos el primer objeto completo y sus mensajes. Para revisar los campos generales creamos una vista tabular en memoria; la variable `conversaciones` conserva la estructura original. Esta revisión de nulos se limita a los campos de la conversación, no a los campos internos de cada mensaje.


In [ ]:
print(f'Tipo: {type(conversaciones).__name__}; conversaciones: {len(conversaciones)}')
print(json.dumps(conversaciones[0], ensure_ascii=False, indent=2))


In [ ]:
conversaciones_df = pd.DataFrame(conversaciones)
display(conversaciones_df.head())
conversaciones_df.info()

display(pd.DataFrame({
    'nulos': conversaciones_df.isna().sum(),
    'porcentaje_nulos': conversaciones_df.isna().mean().mul(100).round(2),
}))
display(conversaciones_df['canal'].value_counts(dropna=False).rename('registros').to_frame())


In [ ]:
# Mensajes de la primera conversación, conservando el orden del archivo.
mensajes_ejemplo = pd.DataFrame(conversaciones[0]['mensajes'])
display(mensajes_ejemplo)
mensajes_ejemplo.info()


## 6. Reglas de normalización

Trabajamos en `datos_normalizados`, un diccionario de copias. Cada columna nueva termina en `_normalizado`, o indica su estado de validación. No eliminamos filas, fusionamos clientes ni sobrescribimos archivos fuente.

- **Espacios:** quitar espacios en los extremos y reducir espacios consecutivos a uno. Una cadena vacía queda como faltante.
- **Nombres:** limpiar únicamente espacios; conservar tildes y mayúsculas originales.
- **Categorías:** usar una clave de comparación sin tildes y en minúsculas para aplicar equivalencias explícitas. Ejemplos: `META ADS` → `Meta Ads`, `sin gestion` → `Sin gestión`, `B/quilla` → `Barranquilla`, `Bogotá D.C.` → `Bogotá`. Mostrar valores no reconocidos.
- **Correos:** limpiar espacios en los extremos y usar minúsculas; esto no valida que la dirección exista.
- **Identificadores:** limpiar espacios y usar mayúsculas. Esto no comprueba duplicados ni relaciones entre tablas.
- **Fechas, teléfonos, modelos y números:** aplicar las reglas documentadas en cada bloque y mostrar los casos pendientes.

In [ ]:
import re
import unicodedata
from datetime import datetime
import math

# Los DataFrames originales permanecen intactos.
datos_normalizados = {nombre: tabla.copy(deep=True) for nombre, tabla in tablas.items()}
datos_normalizados['conversaciones'] = conversaciones_df.copy(deep=True)

def limpiar_texto(valor):
    if pd.isna(valor):
        return pd.NA
    texto = re.sub(r'\s+', ' ', str(valor)).strip()
    return texto if texto else pd.NA

def clave_texto(valor):
    texto = limpiar_texto(valor)
    if pd.isna(texto):
        return None
    return ''.join(c for c in unicodedata.normalize('NFD', texto.casefold())
                   if unicodedata.category(c) != 'Mn')

for nombre, tabla in datos_normalizados.items():
    for columna in tablas[nombre].columns if nombre in tablas else conversaciones_df.columns:
        if columna == 'mensajes':
            continue
        if pd.api.types.is_object_dtype(tabla[columna]) or pd.api.types.is_string_dtype(tabla[columna]):
            tabla[columna + '_normalizado'] = tabla[columna].map(limpiar_texto).astype('string')
            if columna.endswith('_id') or columna == 'sku':
                tabla[columna + '_normalizado'] = tabla[columna + '_normalizado'].str.upper()
            elif columna == 'email':
                tabla[columna + '_normalizado'] = tabla[columna + '_normalizado'].str.lower()

categorias = {
    'canal': {clave_texto(v): v for v in ['WhatsApp', 'Meta Ads', 'Formulario Web']},
    'estado_gestion': {clave_texto(v): v for v in ['Contactado', 'Cotización enviada', 'Descartado', 'En proceso', 'No contesta', 'Sin gestión']},
    'activo': {'si': 'SI', 'no': 'NO'},
    'manifesto_cuota_inicial': {'si': 'SI', 'no': 'NO', 'no_informa': 'NO_INFORMA'},
    'pidio_cita': {'si': 'SI', 'no': 'NO'},
    'forma_pago_declarada': {'contado': 'contado', 'credito': 'credito', 'no_informa': 'no_informa'},
    'desenlace': {clave_texto(v): v for v in ['Cerrado', 'Perdido', 'Sin gestión']},
}
ciudades = ['Barranquilla', 'Bello', 'Bogotá', 'Cartagena', 'Itagüí', 'Medellín', 'Montería', 'Rionegro', 'Santa Marta', 'Soacha', 'Soledad']
categorias['ciudad'] = {clave_texto(v): v for v in ciudades}
categorias['ciudad'].update({
    'b/quilla': 'Barranquilla', 'bogota dc': 'Bogotá', 'bogota d.c.': 'Bogotá',
    'cartagena de indias': 'Cartagena', 'rio negro': 'Rionegro', 'sta marta': 'Santa Marta',
})

for nombre, tabla in datos_normalizados.items():
    for columna, equivalencias in categorias.items():
        if columna not in tabla:
            continue
        claves = tabla[columna].map(clave_texto)
        tabla[columna + '_normalizado'] = claves.map(equivalencias).astype('string')
        tabla[columna + '_estado'] = claves.map(
            lambda clave: 'faltante' if clave is None else ('valido' if clave in equivalencias else 'no_reconocido'))
        if nombre == 'leads':
            print(f'Equivalencias aplicadas: {columna}')
            display(tabla[[columna, columna + '_normalizado', columna + '_estado']].drop_duplicates())

## 7. Teléfonos

Aceptamos números móviles colombianos de 10 dígitos que empiecen por `3`, con prefijo `57` opcional. Quitamos únicamente espacios, guiones, puntos y paréntesis, además de un `+` inicial. La salida válida usa `+57` y los diez dígitos. Letras, longitudes diferentes y otros prefijos se marcan para revisión; no se completan números. Esta es una validación de formato, no de existencia ni de titularidad.

In [ ]:
def normalizar_telefono(valor):
    texto = limpiar_texto(valor)
    if pd.isna(texto):
        return pd.NA, 'faltante'
    if not re.fullmatch(r'\+?[0-9\s().-]+', texto):
        return pd.NA, 'caracteres_no_permitidos'
    digitos = re.sub(r'\D', '', texto)
    if len(digitos) == 12 and digitos.startswith('57'):
        digitos = digitos[2:]
    if len(digitos) != 10:
        return pd.NA, 'longitud_invalida'
    if not digitos.startswith('3'):
        return pd.NA, 'revisar_prefijo_movil'
    return '+57' + digitos, 'valido'

leads_normalizados = datos_normalizados['leads']
leads_normalizados[['telefono_normalizado', 'telefono_estado']] = pd.DataFrame(
    leads['telefono'].map(normalizar_telefono).tolist(), index=leads.index)
display(leads_normalizados[['telefono', 'telefono_normalizado', 'telefono_estado']].head(10))
print('Teléfonos pendientes de revisión:')
display(leads_normalizados.loc[leads_normalizados['telefono_estado'] != 'valido',
    ['lead_id', 'telefono', 'telefono_normalizado', 'telefono_estado']])

## 8. Fechas: corregir el orden de entrada por lead y unificar la salida

**Salida:** siempre año-mes-día (ISO), con hora cuando existe. El orden de entrada se evalúa conjuntamente para registro y primer contacto del **mismo lead**, sin modificar otros clientes.

1. Si el año está al principio, conservar año-mes-día.
2. Con año al final, un valor central de 13 a 31 identifica el día: interpretar mes-día en todas las fechas comparables de ese lead, incluidas las que admiten ambos órdenes.
3. Si el mismo lead tiene evidencia inequívoca de ambos órdenes (por ejemplo `21/08/2026` y `08/21/2026`), reconocer que la fuente ya mezclaba formatos y convertir cada uno a ISO. No invertir una fecha que produciría mes 21. La mezcla y su resolución quedan documentadas.
4. Sin evidencia numérica, usar día-mes. Si el primer contacto resultara anterior al registro y la inversión conjunta del mismo lead recupera la cronología, usarla y marcarla como inferencia por coherencia.
5. Si una fecha inequívoca contradice otra ambigua, conservar la inequívoca y resolver únicamente si una sola combinación de los campos ambiguos recupera la cronología. Se registra como origen mixto inferido.
6. Si persiste una contradicción o la fecha es imposible, conservar el original con un comentario específico. No inventar fechas ni forzar la cronología.

Cada campo tiene `_criterio`; cada lead tiene `orden_fechas_detectado` y `comentario_orden_fechas`. Una fecha sin hora conserva precisión `fecha`: su medianoche técnica no es hora real. No se asigna zona horaria no informada.

Ejemplos: `08/18/2026` se corrige a `2026-08-18`; `2026-08-33` sigue siendo imposible. Las fechas válidas que quedan en una secuencia inconsistente se distinguen de fechas inválidas.


In [ ]:
import calendar

def normalizar_fecha(valor, orden_entrada="DMY"):
    texto = limpiar_texto(valor)
    if pd.isna(texto):
        return pd.NaT, 'faltante', pd.NA, 'La fuente no informa una fecha; no se imputa un valor.'
    inicio = re.fullmatch(r'(\d{4})([-/])(\d{1,2})\2(\d{1,2})(?:[ T](\d{2}):(\d{2})(?::(\d{2}))?)?', texto)
    final = re.fullmatch(r'(\d{1,2})([-/])(\d{1,2})\2(\d{4})(?:[ T](\d{2}):(\d{2})(?::(\d{2}))?)?', texto)
    match = inicio or final
    if not match:
        return pd.NaT, 'invalida', pd.NA, 'Formato no admitido: se requiere AAAA-MM-DD o DD-MM-AAAA (también con barras), con hora HH:MM[:SS] opcional.'
    if inicio:
        anio, mes, dia = int(match[1]), int(match[3]), int(match[4])
        orden = 'año-mes-día'
    else:
        primero, segundo, anio = int(match[1]), int(match[3]), int(match[4])
        dia, mes = (segundo, primero) if orden_entrada == 'MDY' else (primero, segundo)
        orden = 'mes-día-año de origen, corregida a ISO' if orden_entrada == 'MDY' else 'día-mes-año'
    precision = 'fecha_hora' if match[5] is not None else 'fecha'
    if not 1 <= anio <= 9999:
        return pd.NaT, 'invalida', precision, f'Año {anio} fuera del intervalo admitido (1–9999).'
    if not 1 <= mes <= 12:
        return pd.NaT, 'invalida', precision, f'Interpretación {orden}: el mes sería {mes}, fuera de 1–12. El orden se evalúa junto con las otras fechas del mismo lead.'
    maximo = calendar.monthrange(anio, mes)[1]
    if not 1 <= dia <= maximo:
        return pd.NaT, 'invalida', precision, f'Interpretación {orden}: día {dia} imposible; el mes {mes:02d} de {anio} tiene {maximo} días.'
    hora, minuto, segundo = [int(match[i] or 0) for i in (5, 6, 7)]
    if hora > 23 or minuto > 59 or segundo > 59:
        return pd.NaT, 'invalida', precision, f'Hora inválida {hora:02d}:{minuto:02d}:{segundo:02d}; se requiere 00–23:00–59:00–59.'
    valor_normalizado = pd.Timestamp(datetime(anio, mes, dia, hora, minuto, segundo))
    comentario = f'Interpretada como {orden}: {anio:04d}-{mes:02d}-{dia:02d}.'
    if precision == 'fecha':
        comentario += ' La fuente no informa hora; no usar 00:00 como hora real.'
    return valor_normalizado, 'valida', precision, comentario


campos_fecha = {
    'leads': ['fecha_registro', 'fecha_primer_contacto'],
    'asesores': ['fecha_ingreso'],
    'historico_cierres': ['fecha_registro'],
    'conversaciones': ['fecha_inicio'],
}
for nombre, columnas in campos_fecha.items():
    tabla = datos_normalizados[nombre]
    for campo in columnas:
        nuevas = [campo + sufijo for sufijo in ['_normalizado', '_estado', '_precision', '_criterio']]
        tabla[nuevas] = pd.DataFrame(tabla[campo].map(normalizar_fecha).tolist(), index=tabla.index, columns=nuevas)
        tabla[nuevas[0]] = pd.to_datetime(tabla[nuevas[0]])

def evidencia_orden(valor):
    if pd.isna(valor):
        return None
    m = re.match(r'^(\d{1,2})[-/](\d{1,2})[-/]\d{4}(?:\D|$)', str(valor).strip())
    if not m:
        return None  # ISO no cambia de orden.
    a, b = map(int, m.groups())
    if 1 <= a <= 12 and 13 <= b <= 31:
        return 'MDY'
    if 13 <= a <= 31 and 1 <= b <= 12:
        return 'DMY'
    return None

def secuencia_posible(resultados):
    registro, contacto = resultados
    if pd.isna(registro[0]) or pd.isna(contacto[0]):
        return None
    if 'fecha' in [registro[2], contacto[2]]:
        return registro[0].date() <= contacto[0].date()
    return registro[0] <= contacto[0]

tabla = datos_normalizados['leads']
campos_lead = ['fecha_registro', 'fecha_primer_contacto']
tabla['orden_fechas_detectado'] = 'DMY_por_defecto'
tabla['comentario_orden_fechas'] = ''
for indice, fila in tabla.iterrows():
    evidencias = [evidencia_orden(fila[campo]) for campo in campos_lead]
    mixto = 'MDY' in evidencias and 'DMY' in evidencias
    if mixto:
        # Ambos valores tienen evidencia inequívoca. Invertir los dos a ciegas produciría un mes > 12.
        ordenes = evidencias
        regla = 'origen_mixto_resuelto_por_valores'
        motivo = 'El mismo lead combina día-mes y mes-día. Cada valor >12 identifica el día de su campo. Se corrigen ambos a año-mes-día sin convertir un día inequívoco en mes.'
    elif 'MDY' in evidencias:
        ordenes = ['MDY', 'MDY']
        regla = 'MDY_por_evidencia_del_lead'
        motivo = 'Un campo del lead tiene un valor central mayor que 12: es el día. Se interpreta mes-día en todas sus fechas con año al final, incluidas las numéricamente ambiguas. Las ISO no cambian.'
    else:
        ordenes = ['DMY', 'DMY']
        regla = 'DMY_por_evidencia_del_lead' if 'DMY' in evidencias else 'DMY_por_defecto'
        motivo = 'Día-mes-año para fechas con año al final; ISO sin cambios. No hay evidencia de un día en la posición central.'
    resultados = [normalizar_fecha(fila[campo], orden) for campo, orden in zip(campos_lead, ordenes)]
    # Si no hay evidencia numérica y la secuencia contradice el orden por defecto,
    # probar la inversión conjunta del mismo lead, nunca de otros clientes.
    if not any(evidencias) and secuencia_posible(resultados) is False:
        alternativa = [normalizar_fecha(fila[campo], 'MDY') for campo in campos_lead]
        if secuencia_posible(alternativa) is True:
            resultados = alternativa
            regla = 'MDY_por_coherencia_del_lead'
            motivo = 'Sin valores >12 que fijen el orden, día-mes sitúa el contacto antes del registro. La inversión conjunta de las fechas con año al final del mismo lead restaura la secuencia; ISO sin cambios. Inferencia documentada.'

    # Algunos leads mezclan un campo inequívoco con otro de orden ambiguo.
    # Si aún hay contradicción, mantener los campos inequívocos y comparar las
    # alternativas del resto. Resolver solo una combinación cronológica única.
    if secuencia_posible(resultados) is False:
        opciones = []
        for campo, evidencia, actual in zip(campos_lead, evidencias, resultados):
            texto = str(fila[campo]).strip()
            m = re.match(r'^(\d{1,2})[-/](\d{1,2})[-/]\d{4}(?:\D|$)', texto)
            ambiguo = bool(m and all(1 <= int(v) <= 12 for v in m.groups()))
            opciones.append([normalizar_fecha(fila[campo], orden) for orden in ['DMY', 'MDY']]
                           if ambiguo else [actual])
        pares = {}
        for registro in opciones[0]:
            for contacto in opciones[1]:
                if secuencia_posible([registro, contacto]) is True:
                    pares[(registro[0], contacto[0])] = [registro, contacto]
        if len(pares) == 1:
            resultados = next(iter(pares.values()))
            regla = 'origen_mixto_por_coherencia_unica'
            motivo = 'La fuente mezcla órdenes dentro del lead. Manteniendo ISO y los días inequívocos (>12), una sola combinación de sus campos ambiguos sitúa el registro antes del contacto. Se usa esa combinación y se documenta la inferencia; salida ISO en ambos campos.'

    tabla.at[indice, 'orden_fechas_detectado'] = regla
    tabla.at[indice, 'comentario_orden_fechas'] = motivo
    for campo, resultado in zip(campos_lead, resultados):
        fecha, estado, precision, comentario = resultado
        tabla.at[indice, campo + '_normalizado'] = fecha
        tabla.at[indice, campo + '_estado'] = estado
        tabla.at[indice, campo + '_precision'] = precision
        tabla.at[indice, campo + '_criterio'] = comentario + ' ' + motivo

revision_fechas = []
for nombre, columnas in campos_fecha.items():
    tabla = datos_normalizados[nombre]
    identificador = 'conversacion_id' if nombre == 'conversaciones' else ('asesor_id' if nombre == 'asesores' else 'lead_id')
    for campo in columnas:
        for indice, fila in tabla.loc[tabla[campo + '_estado'].eq('invalida')].iterrows():
            revision_fechas.append({'tabla': nombre, 'id': fila[identificador], 'campo': campo,
                'fila_dataframe': indice, 'original': fila[campo], 'estado': 'invalida',
                'normalizado': pd.NaT, 'criterio': fila[campo + '_criterio']})

tabla = datos_normalizados['leads']
tabla['secuencia_fechas_estado'] = 'no_evaluable'
tabla['secuencia_fechas_comentario'] = 'Falta al menos una fecha válida para comparar registro y contacto.'
for indice, fila in tabla.iterrows():
    resultados = [(fila[campo + '_normalizado'], fila[campo + '_estado'], fila[campo + '_precision']) for campo in campos_lead]
    coherente = secuencia_posible(resultados)
    if coherente is None:
        continue
    tabla.at[indice, 'secuencia_fechas_estado'] = 'coherente' if coherente else 'contacto_anterior_registro'
    comentario = ('Registro y primer contacto mantienen orden cronológico con la precisión disponible.' if coherente
                  else f"El contacto ({resultados[1][0]}) sigue siendo anterior al registro ({resultados[0][0]}) tras aplicar la regla por lead. Las fechas son válidas, pero su relación requiere revisión; no se inventa una fecha.")
    tabla.at[indice, 'secuencia_fechas_comentario'] = comentario
    if not coherente:
        revision_fechas.append({'tabla': 'leads', 'id': fila['lead_id'], 'campo': 'fecha_primer_contacto',
            'fila_dataframe': indice, 'original': fila['fecha_primer_contacto'], 'estado': 'secuencia_inconsistente',
            'normalizado': resultados[1][0], 'criterio': comentario})
fechas_por_revisar = pd.DataFrame(revision_fechas, columns=[
    'tabla', 'id', 'campo', 'fila_dataframe', 'original', 'estado', 'normalizado', 'criterio'])
fechas_invalidas = fechas_por_revisar.loc[fechas_por_revisar['estado'].eq('invalida')].copy()
fechas_secuencia_inconsistente = fechas_por_revisar.loc[fechas_por_revisar['estado'].eq('secuencia_inconsistente')].copy()
print('Reglas de interpretación aplicadas por lead:')
display(tabla['orden_fechas_detectado'].value_counts().to_frame('leads'))
print(f'Fechas imposibles o formatos no reconocidos: {len(fechas_invalidas)}')
print(f'Secuencias todavía inconsistentes: {len(fechas_secuencia_inconsistente)}')
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(fechas_por_revisar)
print('Ejemplos de leads corregidos (todas las fechas de salida son ISO):')
display(tabla.loc[tabla['orden_fechas_detectado'].ne('DMY_por_defecto'), [
    'lead_id', 'fecha_registro', 'fecha_registro_normalizado', 'fecha_primer_contacto',
    'fecha_primer_contacto_normalizado', 'orden_fechas_detectado']].head(12))


## 9. Modelos y catálogo

Construimos el nombre canónico con marca y línea del catálogo. Para comparar, limpiamos espacios y diferencias de mayúsculas/tildes, corregimos las marcas observadas `Hnda`, `Bajai`, `Suzuky`, `Heroo` y `A.K.T`, y retiramos un año final de cuatro dígitos únicamente de la clave de búsqueda.

Asignamos coincidencias exactas del nombre completo o de la línea. Una abreviatura se empareja solo si coincide con el inicio de palabras de una única referencia del catálogo y contiene más que la marca; queda marcada como `abreviatura_unica` para distinguir esta inferencia. Si hay varias opciones, mostramos los candidatos sin asignar SKU. No aplicamos similitud difusa ni extraemos información de WhatsApp todavía.

In [ ]:
def clave_modelo(valor):
    clave = clave_texto(valor)
    if clave is None:
        return None
    clave = re.sub(r'^a\.k\.t\b', 'akt', clave)
    for error, marca in {'hnda': 'honda', 'bajai': 'bajaj', 'suzuky': 'suzuki', 'heroo': 'hero'}.items():
        clave = re.sub(r'^' + error + r'\b', marca, clave)
    return re.sub(r'\s+20\d{2}$', '', clave).strip()

referencias = []
for _, moto in catalogo_motos.iterrows():
    nombre = f"{moto['marca']} {moto['linea']}"
    referencias.append({'sku': moto['sku'], 'nombre': nombre,
        'marca': clave_modelo(moto['marca']), 'completo': clave_modelo(nombre),
        'linea': clave_modelo(moto['linea'])})

def emparejar_modelo(valor):
    clave = clave_modelo(valor)
    if clave is None:
        return pd.NA, pd.NA, 'faltante', pd.NA
    candidatos = [r for r in referencias if clave in [r['completo'], r['linea']]]
    estado = 'coincidencia_exacta'
    if not candidatos:
        candidatos = [r for r in referencias if r['completo'].startswith(clave + ' ')
                      or r['linea'].startswith(clave + ' ')]
        estado = 'abreviatura_unica'
    opciones = '; '.join(r['nombre'] for r in candidatos) or pd.NA
    if len(candidatos) == 1 and clave != candidatos[0]['marca']:
        return candidatos[0]['nombre'], candidatos[0]['sku'], estado, opciones
    return pd.NA, pd.NA, 'ambiguo' if candidatos else 'sin_coincidencia', opciones

for nombre, campo in [('leads', 'modelo_interes_texto'), ('historico_cierres', 'modelo_cotizado')]:
    tabla = datos_normalizados[nombre]
    columnas = [campo + '_normalizado', 'modelo_sku', 'modelo_estado', 'modelo_candidatos']
    tabla[columnas] = pd.DataFrame(tabla[campo].map(emparejar_modelo).tolist(), index=tabla.index)
    print(nombre)
    display(tabla['modelo_estado'].value_counts().to_frame('registros'))
    display(tabla[[campo] + columnas].drop_duplicates().head(15))
    print('Modelos pendientes (incluye faltantes):')
    display(tabla.loc[~tabla['modelo_estado'].isin(['coincidencia_exacta', 'abreviatura_unica']),
                      ['lead_id', campo] + columnas])

## 10. Validación de números

Convertimos con `pd.to_numeric`. Marcamos faltantes, valores no numéricos, infinitos, negativos, ceros no permitidos y decimales en campos enteros. Los valores inválidos quedan vacíos en la columna normalizada; el original permanece visible.

Reglas propuestas: precio y cilindraje mayores que cero; unidades disponibles y número de contactos enteros mayores o iguales a cero; capacidad diaria entera mayor que cero; horas hasta el primer contacto mayores o iguales a cero. Un faltante se reporta por separado, sin asumir que es un error. Estas reglas validan tipos y límites básicos, no si un precio es comercialmente correcto.

In [ ]:
reglas_numericas = {
    'catalogo_motos': {'precio_lista': (False, False), 'cilindraje': (True, False), 'unidades_disponibles': (True, True)},
    'asesores': {'capacidad_diaria_leads': (True, False)},
    'historico_cierres': {'precio_lista': (False, False), 'horas_al_primer_contacto': (False, True), 'numero_contactos': (True, True)},
}

def validar_numero(valor, entero, permite_cero):
    if pd.isna(valor) or not str(valor).strip():
        return pd.NA, 'faltante'
    numero = pd.to_numeric(valor, errors='coerce')
    if pd.isna(numero):
        return pd.NA, 'no_numerico'
    if not math.isfinite(float(numero)):
        return pd.NA, 'no_finito'
    if numero < 0:
        return pd.NA, 'negativo'
    if numero == 0 and not permite_cero:
        return pd.NA, 'cero_no_permitido'
    if entero and numero % 1 != 0:
        return pd.NA, 'debe_ser_entero'
    return numero, 'valido'

revision_numerica = []
for nombre, reglas in reglas_numericas.items():
    tabla = datos_normalizados[nombre]
    identificador = {'catalogo_motos': 'sku', 'asesores': 'asesor_id', 'historico_cierres': 'lead_id'}[nombre]
    for campo, (entero, permite_cero) in reglas.items():
        resultado = tabla[campo].map(lambda v: validar_numero(v, entero, permite_cero))
        tabla[campo + '_normalizado'] = pd.Series([v[0] for v in resultado], index=tabla.index,
                                                  dtype='Int64' if entero else 'Float64')
        tabla[campo + '_estado'] = [v[1] for v in resultado]
        for indice in tabla.index[tabla[campo + '_estado'] != 'valido']:
            revision_numerica.append({'tabla': nombre, 'fila_dataframe': indice,
                'id': tabla.at[indice, identificador], 'campo': campo,
                'original': tabla.at[indice, campo], 'estado': tabla.at[indice, campo + '_estado']})

numeros_por_revisar = pd.DataFrame(revision_numerica,
    columns=['tabla', 'fila_dataframe', 'id', 'campo', 'original', 'estado'])
numeros_invalidos = numeros_por_revisar.loc[numeros_por_revisar['estado'] != 'faltante']
numeros_faltantes = numeros_por_revisar.loc[numeros_por_revisar['estado'] == 'faltante']
print(f'Valores numéricos inválidos: {len(numeros_invalidos)}')
with pd.option_context('display.max_rows', None):
    display(numeros_invalidos)
print(f'Valores numéricos faltantes: {len(numeros_faltantes)}')
with pd.option_context('display.max_rows', None):
    display(numeros_faltantes)

## 11. Resumen de validaciones

Los conteos siguientes son por campo, no por persona: una fila puede tener varios pendientes. `faltante` no equivale a `inválido`. Las fechas que no cumplen la regla acordada y las secuencias inconsistentes quedan pendientes; las abreviaturas de modelos conservan su marca de inferencia. Los originales siguen disponibles en `leads`, `asesores`, `catalogo_motos`, `historico_cierres` y `conversaciones`.

In [ ]:
conteos_validacion = []
for nombre, tabla in datos_normalizados.items():
    for campo in [c for c in tabla.columns if c.endswith('_estado')]:
        for estado, cantidad in tabla[campo].value_counts(dropna=False).items():
            conteos_validacion.append({'tabla': nombre, 'validacion': campo, 'estado': estado, 'registros': cantidad})
resumen_validaciones = pd.DataFrame(conteos_validacion)
with pd.option_context('display.max_rows', None):
    display(resumen_validaciones)
print('Categorías no reconocidas:')
for nombre, tabla in datos_normalizados.items():
    for campo in categorias:
        if campo + '_estado' in tabla:
            pendientes = tabla.loc[tabla[campo + '_estado'] == 'no_reconocido', [campo, campo + '_estado']]
            if not pendientes.empty:
                print(nombre, campo)
                display(pendientes)

## 12. Candidatos a duplicados dentro de cada empresa

Agrupamos por **empresa y teléfono normalizados**, usando únicamente teléfonos válidos y empresas informadas. Una coincidencia identifica un candidato a revisión, no confirma que sea la misma persona ni que sus consultas deban fusionarse. No agrupamos clientes entre empresas.

Para cada grupo mostramos los IDs, las filas del DataFrame original, nombres, correos, canales, fechas y modelos. Contrastamos nombres ignorando espacios, mayúsculas y tildes, pero **no interpretamos automáticamente iniciales**. Los correos faltantes no cuentan como coincidencia ni como diferencia.

Distinguimos filas completamente idénticas de registros con diferencias. La marca de fila idéntica se calcula sobre todas las columnas originales, antes de añadir las columnas de diagnóstico. Un interés en otra moto o una fecha distinta puede representar una nueva consulta válida. No eliminamos ni fusionamos registros.

In [ ]:
# Copia exclusiva para el diagnóstico; las tablas originales y normalizadas no se modifican.
revision_duplicados = datos_normalizados['leads'].copy(deep=True)
revision_duplicados['fila_dataframe'] = revision_duplicados.index
revision_duplicados['fila_original_repetida'] = leads.duplicated(keep=False)
revision_duplicados['nombre_clave_comparacion'] = revision_duplicados['nombre_cliente'].map(clave_texto)

claves_duplicados = ['empresa_id_normalizado', 'telefono_normalizado']
elegibles = (revision_duplicados['telefono_estado'].eq('valido')
             & revision_duplicados['empresa_id_normalizado'].notna()
             & revision_duplicados['telefono_normalizado'].notna())
base_duplicados = revision_duplicados.loc[elegibles].copy()
candidatos_duplicados = base_duplicados.loc[
    base_duplicados.duplicated(subset=claves_duplicados, keep=False)
].copy()
candidatos_duplicados = candidatos_duplicados.sort_values(claves_duplicados + ['fila_dataframe'])
candidatos_duplicados['grupo_duplicado'] = (
    candidatos_duplicados.groupby(claves_duplicados, sort=True).ngroup() + 1
).map(lambda numero: f'DUP-{numero:03d}')

resumen_grupos = []
for grupo_id, grupo in candidatos_duplicados.groupby('grupo_duplicado', sort=True):
    nombres = grupo['nombre_clave_comparacion'].nunique(dropna=True)
    correos = grupo['email_normalizado'].nunique(dropna=True)
    nombres_faltantes = int(grupo['nombre_clave_comparacion'].isna().sum())
    correos_faltantes = int(grupo['email_normalizado'].isna().sum())
    filas_distintas = len(grupo[list(leads.columns)].drop_duplicates())
    resumen_grupos.append({
        'grupo': grupo_id,
        'empresa': grupo['empresa_id_normalizado'].iloc[0],
        'telefono': grupo['telefono_normalizado'].iloc[0],
        'filas': len(grupo),
        'ids_distintos': grupo['lead_id'].nunique(),
        'filas_originales_distintas': filas_distintas,
        'filas_en_repeticiones_exactas': int(grupo['fila_original_repetida'].sum()),
        'nombres_distintos': nombres,
        'nombres_faltantes': nombres_faltantes,
        'correos_distintos': correos,
        'correos_faltantes': correos_faltantes,
        'canales_distintos': grupo['canal_normalizado'].nunique(dropna=True),
        'lectura_inicial': (
            'Todas las filas son idénticas' if filas_distintas == 1
            else 'Revisar diferencias de nombre o correo' if nombres > 1 or correos > 1
            else 'Datos de identidad compatibles; revisar consultas y faltantes'
        ),
    })
resumen_duplicados = pd.DataFrame(resumen_grupos, columns=[
    'grupo', 'empresa', 'telefono', 'filas', 'ids_distintos', 'filas_originales_distintas',
    'filas_en_repeticiones_exactas', 'nombres_distintos', 'nombres_faltantes',
    'correos_distintos', 'correos_faltantes', 'canales_distintos', 'lectura_inicial',
])

print(f'Grupos candidatos: {len(resumen_duplicados)}')
print(f'Filas incluidas: {len(candidatos_duplicados)}')
print(f'Filas excluidas por teléfono inválido/faltante o empresa faltante: {(~elegibles).sum()}')
print('Los IDs repetidos no se cuentan como personas distintas.')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(resumen_duplicados)

### Comparación de los registros candidatos

Las dos tablas comparten `grupo_duplicado` y `fila_dataframe` para ubicar cada registro sin depender de que `lead_id` sea único. La primera muestra identidad y canal; la segunda, fechas y modelos con sus estados de normalización. Un `NaT` acompañado de `invalida` significa que la fecha original no cumple la regla acordada; `_criterio` explica el motivo.

In [ ]:
columnas_identidad = [
    'grupo_duplicado', 'fila_dataframe', 'lead_id', 'empresa_id_normalizado',
    'telefono', 'telefono_normalizado', 'nombre_cliente', 'nombre_clave_comparacion',
    'email', 'email_normalizado', 'canal_normalizado', 'punto_venta_id_normalizado',
    'fila_original_repetida',
]
columnas_consulta = [
    'grupo_duplicado', 'fila_dataframe', 'lead_id',
    'fecha_registro', 'fecha_registro_normalizado', 'fecha_registro_estado',
    'fecha_primer_contacto', 'fecha_primer_contacto_normalizado', 'fecha_primer_contacto_estado',
    'modelo_interes_texto', 'modelo_interes_texto_normalizado', 'modelo_estado',
    'estado_gestion_normalizado',
]
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', 80):
    print('Identidad y canal:')
    display(candidatos_duplicados[columnas_identidad])
    print('Fechas, modelos y estado comercial:')
    display(candidatos_duplicados[columnas_consulta])

### Inspeccionar un grupo

Cambia `grupo_a_revisar` por uno de los códigos del resumen. El filtro solo muestra información; no modifica registros ni confirma duplicados. Los códigos se generan para esta ejecución y no son identificadores permanentes de clientes.

In [ ]:
grupo_a_revisar = 'DUP-001'
detalle_grupo = candidatos_duplicados.loc[candidatos_duplicados['grupo_duplicado'] == grupo_a_revisar]
with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
    display(detalle_grupo[columnas_identidad])
    display(detalle_grupo[columnas_consulta])

## 13. Validación de relaciones entre archivos

Revisamos si los identificadores permiten conectar los archivos sin perder información ni multiplicar filas. Este bloque crea tablas de diagnóstico; no elimina, fusiona ni asigna clientes.

1. Revisar IDs repetidos o faltantes antes de hacer cruces.
2. Comprobar que cada conversación tenga un lead y una empresa identificable.
3. Contrastar empresa y punto de venta de los leads con la relación observada en asesores.
4. Revisar cobertura de asesores activos y disponibilidad del modelo en el punto asignado.

**Límite de la fuente:** no hay un maestro independiente de empresas y puntos. Usamos la relación de `asesores.csv` como referencia operativa, no como prueba externa de que la asignación es correcta. Los IDs del histórico corresponden a otros leads y no tienen por qué aparecer en el archivo actual.

In [ ]:
# Primero, comprobar las claves de cada tabla.
claves_por_tabla = {
    'leads': 'lead_id_normalizado', 'asesores': 'asesor_id_normalizado',
    'catalogo_motos': 'sku_normalizado', 'historico_cierres': 'lead_id_normalizado',
    'conversaciones': 'conversacion_id_normalizado',
}
revision_ids = []
ids_repetidos = {}
for nombre, clave in claves_por_tabla.items():
    origen = datos_normalizados[nombre]
    repetidos = origen[clave].notna() & origen[clave].duplicated(keep=False)
    ids_repetidos[nombre] = origen.loc[repetidos].copy()
    revision_ids.append({'tabla': nombre, 'clave': clave,
        'ids_faltantes': int(origen[clave].isna().sum()),
        'ids_repetidos_distintos': origen.loc[repetidos, clave].nunique(),
        'filas_con_id_repetido': int(repetidos.sum())})
display(pd.DataFrame(revision_ids))
print('Leads con ID repetido:')
display(ids_repetidos['leads'][list(leads.columns)])

### Conversaciones y leads

Para determinar pertenencia, contamos las filas y empresas asociadas a cada `lead_id` normalizado. Este resumen tiene una fila por ID y evita que una conversación se duplique durante el cruce. **No es una consolidación del cliente**: las filas originales siguen intactas.

Una conversación sin lead queda pendiente. Si un ID de lead aparece en varias empresas o en una fila sin empresa, no asignamos empresa a la conversación. Aunque se determine una sola empresa, un ID repetido se marca para revisar antes de unir otros atributos del lead.

In [ ]:
leads_relaciones = datos_normalizados['leads']
conversaciones_relaciones = datos_normalizados['conversaciones']
referencia_leads = leads_relaciones.loc[leads_relaciones['lead_id_normalizado'].notna()].groupby(
    'lead_id_normalizado', as_index=False
).agg(
    filas_lead=('lead_id_normalizado', 'size'),
    empresas_distintas=('empresa_id_normalizado', 'nunique'),
    empresas_faltantes=('empresa_id_normalizado', lambda s: int(s.isna().sum())),
    empresa_identificada=('empresa_id_normalizado', lambda s: s.dropna().iloc[0]
                         if s.nunique() == 1 and s.notna().all() else pd.NA),
)
revision_conversaciones = conversaciones_relaciones[
    ['conversacion_id_normalizado', 'lead_id_normalizado', 'fecha_inicio_normalizado']
].merge(referencia_leads, on='lead_id_normalizado', how='left', validate='many_to_one', indicator=True)
revision_conversaciones['estado_relacion'] = 'lead_unico'
revision_conversaciones.loc[revision_conversaciones['filas_lead'].gt(1), 'estado_relacion'] = 'id_lead_repetido'
revision_conversaciones.loc[
    revision_conversaciones['_merge'].eq('both') & revision_conversaciones['empresa_identificada'].isna(),
    'estado_relacion'] = 'empresa_no_determinable'
revision_conversaciones.loc[revision_conversaciones['_merge'].eq('left_only'), 'estado_relacion'] = 'sin_lead'
conversaciones_sin_lead = revision_conversaciones.loc[revision_conversaciones['estado_relacion'].eq('sin_lead')].copy()
print('Estado del vínculo de cada conversación:')
display(revision_conversaciones['estado_relacion'].value_counts().to_frame('conversaciones'))
print('Conversaciones que requieren revisión:')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(revision_conversaciones.loc[revision_conversaciones['estado_relacion'].ne('lead_unico')])

### Empresa, punto de venta y asesores

Comparamos pares completos `(empresa, punto de venta)`. Comprobar solo que el punto exista no detectaría un lead asignado a un punto de otra empresa.

También contamos asesores activos por par y sumamos sus capacidades declaradas válidas. Esa suma representa capacidad teórica diaria, no disponibilidad real: no tenemos agenda, ausencias ni asignaciones actuales. Si un asesor activo tiene capacidad inválida o faltante, dejamos el total del punto sin calcular. Si se repite un ID de asesor, detenemos esta suma para evitar contar dos veces a la misma persona.

In [ ]:
asesores_relaciones = datos_normalizados['asesores']
claves_punto = ['empresa_id_normalizado', 'punto_venta_id_normalizado']
pares_puntos = asesores_relaciones[claves_punto].dropna().drop_duplicates()
revision_puntos = leads_relaciones[['lead_id_normalizado'] + claves_punto].copy()
revision_puntos.insert(0, 'fila_dataframe', leads_relaciones.index)
revision_puntos = revision_puntos.merge(pares_puntos, on=claves_punto, how='left', validate='many_to_one', indicator=True)
leads_punto_inconsistente = revision_puntos.loc[revision_puntos['_merge'].eq('left_only')].copy()
empresas_por_punto = pares_puntos.groupby('punto_venta_id_normalizado')['empresa_id_normalizado'].nunique()
puntos_varias_empresas = empresas_por_punto.loc[empresas_por_punto.gt(1)]
print('Leads cuyo par empresa/punto no aparece en asesores:')
display(leads_punto_inconsistente)
print('Puntos asociados a varias empresas en asesores:')
display(puntos_varias_empresas.to_frame('empresas'))

if asesores_relaciones['asesor_id_normalizado'].isna().any() or asesores_relaciones['asesor_id_normalizado'].duplicated().any():
    raise ValueError('Revisar IDs de asesores antes de calcular cobertura y capacidad.')
cobertura_filas = []
for (empresa, punto), grupo in asesores_relaciones.groupby(claves_punto, dropna=False):
    activos = grupo.loc[grupo['activo_normalizado'].eq('SI')]
    capacidad_valida = activos['capacidad_diaria_leads_estado'].eq('valido').all()
    cobertura_filas.append({'empresa': empresa, 'punto': punto, 'asesores': len(grupo),
        'activos': len(activos), 'estado_activo_desconocido': int(grupo['activo_normalizado'].isna().sum()),
        'capacidad_teorica_activos': activos['capacidad_diaria_leads_normalizado'].sum() if capacidad_valida else pd.NA})
cobertura_asesores = pd.DataFrame(cobertura_filas)
display(cobertura_asesores)

### Catálogo y disponibilidad por punto

Separamos la lista de puntos del catálogo en una tabla auxiliar para comprobar que todos existan en la referencia de asesores. Luego contrastamos el SKU identificado para cada lead con su punto de venta.

Clasificamos cada caso como disponible, no listado en el punto o no evaluable. **No listado** es una observación comercial, no necesariamente un error de datos: podría requerir traslado o una alternativa de producto. Las asignaciones por abreviatura mantienen su marca de inferencia. No distribuimos `unidades_disponibles` entre puntos, porque el archivo no entrega ese desglose.

In [ ]:
catalogo_relaciones = datos_normalizados['catalogo_motos']
if catalogo_relaciones['sku_normalizado'].isna().any() or catalogo_relaciones['sku_normalizado'].duplicated().any():
    raise ValueError('Revisar SKUs antes de comprobar disponibilidad.')
disponibilidad_catalogo = catalogo_relaciones[['sku_normalizado', 'puntos_venta_disponibles_normalizado']].copy()
disponibilidad_catalogo['punto'] = disponibilidad_catalogo['puntos_venta_disponibles_normalizado'].str.split('|', regex=False)
disponibilidad_catalogo = disponibilidad_catalogo.explode('punto')
disponibilidad_catalogo['punto'] = disponibilidad_catalogo['punto'].astype('string').str.strip().str.upper()
puntos_catalogo_desconocidos = disponibilidad_catalogo.loc[
    ~disponibilidad_catalogo['punto'].isin(pares_puntos['punto_venta_id_normalizado'])].copy()
print('Puntos del catálogo que no aparecen en asesores:')
display(puntos_catalogo_desconocidos)

pares_disponibilidad = set(disponibilidad_catalogo[['sku_normalizado', 'punto']].dropna().itertuples(index=False, name=None))
pares_empresa_punto = set(pares_puntos.itertuples(index=False, name=None))
skus_conocidos = set(catalogo_relaciones['sku_normalizado'].dropna())
revision_disponibilidad = leads_relaciones[[
    'lead_id_normalizado', 'empresa_id_normalizado', 'punto_venta_id_normalizado',
    'modelo_interes_texto', 'modelo_interes_texto_normalizado', 'modelo_sku', 'modelo_estado',
]].copy()
revision_disponibilidad.insert(0, 'fila_dataframe', leads_relaciones.index)

def comprobar_disponibilidad(fila):
    if pd.isna(fila['modelo_sku']):
        return 'no_evaluable_modelo_pendiente'
    if fila['modelo_sku'] not in skus_conocidos:
        return 'no_evaluable_sku_desconocido'
    empresa, punto = fila['empresa_id_normalizado'], fila['punto_venta_id_normalizado']
    if pd.isna(empresa) or pd.isna(punto) or (empresa, punto) not in pares_empresa_punto:
        return 'no_evaluable_empresa_punto'
    return 'listado_en_punto' if (fila['modelo_sku'], punto) in pares_disponibilidad else 'no_listado_en_punto'

revision_disponibilidad['disponibilidad'] = revision_disponibilidad.apply(comprobar_disponibilidad, axis=1)
leads_modelo_no_disponible = revision_disponibilidad.loc[revision_disponibilidad['disponibilidad'].eq('no_listado_en_punto')].copy()
display(revision_disponibilidad['disponibilidad'].value_counts().to_frame('filas_lead'))
print('Primeros 20 casos no listados en el punto. Todos quedan en leads_modelo_no_disponible.')
with pd.option_context('display.max_columns', None):
    display(leads_modelo_no_disponible.head(20))

### Cómo interpretar y atender los hallazgos

- **Conversación sin lead:** conservar pendiente de vinculación; solicitar el registro faltante o confirmar la referencia. No inventar empresa ni cliente.
- **ID de lead repetido:** revisar las filas antes de unir atributos. Un resumen para comprobar pertenencia no reemplaza esa decisión.
- **Par empresa/punto desconocido:** revisar contra un maestro de puntos antes de asignar asesor.
- **Modelo no listado en el punto:** conservar el interés expresado; verificar disponibilidad comercial sin sustituir automáticamente el modelo.
- **Modelo pendiente:** mantenerlo sin SKU hasta contar con evidencia suficiente.

El siguiente paso será definir las reglas de consolidación de los candidatos a duplicados. Hasta entonces, estas tablas son diagnósticos y mantienen todas las filas originales.

## 14. Consolidación conservadora y trazabilidad

Distinguimos **una fila repetida**, **una consulta comercial** y **un cliente**. Una persona puede hacer varias consultas, interesarse por distintas motos y escribir en distintos momentos. Por eso no basta con coincidir en teléfono para borrar una consulta.

### Reglas aplicadas

1. **Filas idénticas en todas las columnas originales:** conservar la primera aparición en una nueva tabla `leads_sin_repeticiones_exactas`. Registrar el vínculo de cada fila con la conservada. No usar solo el teléfono ni el ID para eliminar filas.
2. **Misma empresa y teléfono válido, con varias filas diferentes:** conservar todas las consultas y generar una propuesta de agrupación de cliente.
3. **Nombre completo coincidente y correos sin contradicción:** proponer agrupar bajo un cliente, sujeto a revisión. Un correo faltante no prueba identidad. Ignoramos mayúsculas, tildes y espacios para comparar nombres, pero no expandimos iniciales.
4. **Nombres diferentes, correos diferentes o nombres faltantes:** dejar el grupo pendiente. Estas diferencias pueden ser errores de captura, abreviaturas o personas que comparten teléfono; no decidimos automáticamente.
5. **Empresas distintas:** mantener la separación. Las propuestas usan siempre empresa y teléfono como clave conjunta.

Este bloque aplica únicamente la regla 1 sobre una copia en memoria. Las demás son propuestas visibles, no fusiones de clientes. Las fechas rechazadas y los modelos pendientes se conservan con sus marcas.

In [ ]:
# Comparación exacta sobre todas las columnas originales, incluidos los faltantes.
# La primera aparición es solo un criterio estable de conservación, no una preferencia comercial.
grupos_fila_exacta = leads.groupby(list(leads.columns), dropna=False, sort=False).ngroup()
primera_fila_por_grupo = pd.Series(leads.index, index=leads.index).groupby(grupos_fila_exacta).first()
filas_conservadas = grupos_fila_exacta.map(primera_fila_por_grupo)

trazabilidad_filas = pd.DataFrame({
    'fila_origen': leads.index,
    'lead_id_origen': leads['lead_id'],
    'empresa_origen': leads['empresa_id'],
    'fila_conservada': filas_conservadas,
})
trazabilidad_filas['accion'] = 'conservar'
trazabilidad_filas.loc[trazabilidad_filas['fila_origen'].ne(trazabilidad_filas['fila_conservada']), 'accion'] = 'repeticion_exacta'
mascara_conservar = ~leads.duplicated(keep='first')
leads_sin_repeticiones_exactas = datos_normalizados['leads'].loc[mascara_conservar].copy(deep=True)
leads_sin_repeticiones_exactas.insert(0, 'fila_origen', leads_sin_repeticiones_exactas.index)

print(f'Filas originales: {len(leads)}')
print(f'Filas conservadas en la copia: {len(leads_sin_repeticiones_exactas)}')
print(f'Repeticiones exactas consolidadas: {(~mascara_conservar).sum()}')
print('Trazabilidad de las filas afectadas (incluye la fila conservada):')
filas_repetidas = leads.duplicated(keep=False)
display(trazabilidad_filas.loc[filas_repetidas])
print('Contenido original de esas filas:')
display(leads.loc[filas_repetidas])

### Propuestas para clientes con varias consultas

Volvemos a buscar teléfonos repetidos dentro de cada empresa sobre la copia sin repeticiones exactas. El resumen muestra la evidencia, los IDs de todas las consultas y los motivos de revisión. Los códigos de grupo sirven para navegar este análisis; no se convierten en IDs definitivos de cliente.

Incluso cuando la identidad parece compatible, la propuesta es **agrupar al cliente manteniendo sus consultas**, no elegir una fila y perder las demás. No elegimos un único modelo de interés, no sobrescribimos estados comerciales y no trasladamos conversaciones de un ID a otro.

In [ ]:
base_propuestas = leads_sin_repeticiones_exactas.loc[
    leads_sin_repeticiones_exactas['telefono_estado'].eq('valido')
    & leads_sin_repeticiones_exactas['empresa_id_normalizado'].notna()
].copy()
base_propuestas['nombre_clave_comparacion'] = base_propuestas['nombre_cliente'].map(clave_texto)
claves_cliente = ['empresa_id_normalizado', 'telefono_normalizado']
consultas_candidatas = base_propuestas.loc[base_propuestas.duplicated(claves_cliente, keep=False)].copy()
consultas_candidatas = consultas_candidatas.sort_values(claves_cliente + ['fila_origen'])
consultas_candidatas['grupo_propuesto'] = (
    consultas_candidatas.groupby(claves_cliente, sort=True).ngroup() + 1
).map(lambda numero: f'CLI-PROP-{numero:03d}')

propuestas = []
for codigo, grupo in consultas_candidatas.groupby('grupo_propuesto', sort=True):
    nombres = grupo['nombre_clave_comparacion'].nunique(dropna=True)
    correos = grupo['email_normalizado'].nunique(dropna=True)
    nombres_faltantes = grupo['nombre_clave_comparacion'].isna().any()
    # Un nombre con inicial aislada o una sola palabra no cuenta como nombre completo.
    nombre_completo = grupo['nombre_clave_comparacion'].map(
        lambda v: isinstance(v, str) and len(v.split()) >= 2
        and all(len(parte.strip('.')) > 1 for parte in v.split())
    ).all()
    identidad_compatible = nombres == 1 and not nombres_faltantes and nombre_completo and correos <= 1
    motivos = []
    if nombres > 1:
        motivos.append('nombres diferentes; pueden incluir abreviaturas')
    if nombres_faltantes:
        motivos.append('nombre faltante')
    if not nombre_completo:
        motivos.append('nombre incompleto o con iniciales')
    if correos > 1:
        motivos.append('correos diferentes')
    if grupo['email_normalizado'].isna().any():
        motivos.append('correo faltante en alguna consulta')
    if identidad_compatible:
        motivos.insert(0, 'mismo nombre completo normalizado y sin correos contradictorios')
    propuestas.append({
        'grupo': codigo,
        'empresa': grupo['empresa_id_normalizado'].iloc[0],
        'telefono': grupo['telefono_normalizado'].iloc[0],
        'consultas': len(grupo),
        'lead_ids': ', '.join(grupo['lead_id'].astype(str)),
        'nombres_originales': ' | '.join(grupo['nombre_cliente'].dropna().astype(str).unique()),
        'correos_normalizados': ' | '.join(grupo['email_normalizado'].dropna().unique()),
        'propuesta': 'agrupar_cliente_conservar_consultas' if identidad_compatible else 'revision_manual_identidad',
        'motivo': '; '.join(motivos) or 'revisar evidencia de identidad',
        'consultas_con_fecha_pendiente': int(grupo['fecha_registro_estado'].isin(['ambigua', 'invalida', 'faltante']).sum()),
        'consultas_con_modelo_pendiente': int((~grupo['modelo_estado'].isin(['coincidencia_exacta', 'abreviatura_unica'])).sum()),
    })
propuestas_clientes = pd.DataFrame(propuestas)
print('Propuestas de agrupación (ninguna se aplica automáticamente):')
display(propuestas_clientes['propuesta'].value_counts().to_frame('grupos'))
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', 100):
    display(propuestas_clientes)

### Comprobar que las conversaciones mantienen su vínculo

La consolidación exacta conserva los mismos IDs de lead, por lo que no requiere reasignar conversaciones. Verificamos de nuevo los IDs y cruzamos únicamente cuando son únicos y tienen empresa informada. Si la clave sigue repetida, detenemos el cruce para revisarla.

Este resultado se guarda en `conversaciones_con_lead_consolidado`, una vista auxiliar. El JSON original permanece intacto; las conversaciones sin lead siguen pendientes, sin empresa asignada.

In [ ]:
referencia_consolidada = leads_sin_repeticiones_exactas[
    ['lead_id_normalizado', 'empresa_id_normalizado', 'fila_origen']
].copy()
if referencia_consolidada['lead_id_normalizado'].isna().any() or referencia_consolidada['lead_id_normalizado'].duplicated().any():
    raise ValueError('Quedan IDs de lead faltantes o repetidos; revisar antes de unir conversaciones.')
if referencia_consolidada['empresa_id_normalizado'].isna().any():
    raise ValueError('Hay leads sin empresa; revisar antes de asignar pertenencia a conversaciones.')

conversaciones_con_lead_consolidado = datos_normalizados['conversaciones'][
    ['conversacion_id_normalizado', 'lead_id_normalizado', 'fecha_inicio_normalizado']
].merge(referencia_consolidada, on='lead_id_normalizado', how='left', validate='many_to_one', indicator=True)
print('Conversaciones con vínculo a lead después de consolidar únicamente filas idénticas:')
display(conversaciones_con_lead_consolidado['_merge'].value_counts().rename(index={
    'both': 'con_lead', 'left_only': 'sin_lead', 'right_only': 'no_aplica'
}).to_frame('conversaciones'))
print('Conversaciones todavía sin lead:')
display(conversaciones_con_lead_consolidado.loc[conversaciones_con_lead_consolidado['_merge'].eq('left_only')])

### Estado al terminar este bloque

- `leads` y `datos_normalizados['leads']`: mantienen todas las filas originales.
- `leads_sin_repeticiones_exactas`: copia que conserva una fila por contenido original idéntico.
- `trazabilidad_filas`: permite rastrear cada fila recibida a su fila conservada.
- `propuestas_clientes` y `consultas_candidatas`: evidencia para revisar posibles clientes con varias consultas. No se ha creado un cliente único por teléfono.
- `conversaciones_con_lead_consolidado`: verifica los vínculos tras quitar repeticiones exactas, sin alterar mensajes ni asignar los casos huérfanos.

No se exportan archivos limpios todavía. Antes de llevar la consolidación al flujo automático, habrá que decidir qué propuestas de identidad aceptar y cómo mantener consultas, intereses y conversaciones separados dentro de un mismo cliente.

## 15. Resolución automática de duplicados y salida consolidada

### Regla de identidad elegida para este ejercicio

Agrupar consultas cuando tengan **la misma empresa, el mismo teléfono móvil válido y nombres compatibles**. No usar el correo como identificador único ni descartar correos diferentes: una persona puede declarar más de uno. Todos los correos se conservan y sus diferencias se marcan.

Los nombres son compatibles si coinciden tras normalizar espacios, tildes y mayúsculas, o si el más corto corresponde a una abreviatura del largo: mismo primer nombre (o su inicial), al menos otro término completo coincidente y términos restantes presentes en el mismo orden. Ejemplos: `Y. Castaño Valencia` con `Yuliana Castaño Valencia`; `María Valencia` con `María Fernanda Valencia Salazar`. No se corrigen nombres por similitud de letras.

La compatibilidad debe cumplirse entre **todos los pares** del grupo para evitar uniones transitivas sin evidencia. Si no se cumple, las consultas permanecen separadas y el grupo queda como excepción de identidad. Un teléfono inválido no sirve para agrupar.

**Supuesto y limitación:** el teléfono compartido y el nombre compatible se consideran evidencia suficiente para este ejercicio. No garantizan identidad real; podrían existir teléfonos familiares y nombres parecidos. La decisión queda registrada y es reversible mediante el mapa de origen.

### Qué significa consolidar

`leads_consolidados` contiene una fila por identidad resuelta dentro de una empresa. `consultas_resueltas` conserva una fila por consulta distinta, con sus fechas, estado, canal e interés originales y normalizados. No se elige arbitrariamente una única moto ni un único correo. `mapa_origen_consolidado` enlaza todas las filas recibidas con su identidad consolidada, incluidas las repeticiones exactas.

In [ ]:
import hashlib
from itertools import combinations

def nombres_compatibles(nombre_a, nombre_b):
    a, b = clave_texto(nombre_a), clave_texto(nombre_b)
    if not a or not b:
        return False
    a = [t.strip('.') for t in a.split()]
    b = [t.strip('.') for t in b.split()]
    if len(a) < 2 or len(b) < 2:
        return False
    if a == b:
        return all(len(t) > 1 for t in a)
    corto, largo = sorted([a, b], key=lambda partes: (len(partes), sum(map(len, partes))))
    if any(len(t) <= 1 for t in largo):
        return False
    primero_coincide = corto[0] == largo[0] or (len(corto[0]) == 1 and largo[0].startswith(corto[0]))
    if not primero_coincide or not any(len(t) > 1 for t in corto[1:]):
        return False
    posicion = 1
    for termino in corto[1:]:
        # Solo aceptamos inicial en el primer término.
        if len(termino) <= 1:
            return False
        while posicion < len(largo) and largo[posicion] != termino:
            posicion += 1
        if posicion == len(largo):
            return False
        posicion += 1
    return True

def id_consolidado(empresa, referencia):
    # ID reproducible que no muestra el teléfono; no es anonimización criptográfica.
    contenido = json.dumps([empresa, referencia], ensure_ascii=False)
    return 'LC-' + hashlib.sha256(contenido.encode('utf-8')).hexdigest()[:20]

consultas_resueltas = leads_sin_repeticiones_exactas.copy(deep=True)
consultas_resueltas['lead_consolidado_id'] = [
    id_consolidado(str(fila['empresa_id_normalizado']), 'consulta:' + str(fila['lead_id_normalizado']))
    for _, fila in consultas_resueltas.iterrows()
]
consultas_resueltas['regla_identidad'] = 'consulta_individual'
decisiones_identidad = []
for codigo, grupo in consultas_candidatas.groupby('grupo_propuesto', sort=True):
    compatibles = all(nombres_compatibles(a, b) for a, b in combinations(grupo['nombre_cliente'].tolist(), 2))
    empresa = grupo['empresa_id_normalizado'].iloc[0]
    telefono = grupo['telefono_normalizado'].iloc[0]
    if compatibles:
        identificador = id_consolidado(str(empresa), 'telefono:' + telefono)
        consultas_resueltas.loc[grupo.index, 'lead_consolidado_id'] = identificador
        consultas_resueltas.loc[grupo.index, 'regla_identidad'] = 'empresa_telefono_nombre_compatible'
    decisiones_identidad.append({
        'grupo': codigo, 'empresa': empresa, 'telefono': telefono,
        'lead_ids': grupo['lead_id_normalizado'].tolist(),
        'consultas': len(grupo), 'decision': 'agrupado' if compatibles else 'separado_por_identidad_incierta',
        'correos_distintos': grupo['email_normalizado'].nunique(dropna=True),
        'nombres_originales': grupo['nombre_cliente'].tolist(),
    })
decisiones_identidad = pd.DataFrame(decisiones_identidad)
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 100):
    display(decisiones_identidad)

### Una identidad, todas sus consultas

El nombre de presentación será la variante informada más completa (más palabras completas y luego mayor longitud). Es un criterio de presentación: se conservan también las demás variantes. Las fechas agregadas usan únicamente valores resueltos y muestran cuántas fechas quedaron pendientes; no representan una cronología completa si existe esa marca.

Cuando una identidad tiene un teléfono inválido o faltante, se mantiene en la salida con una marca que impide tratarla como lista para contactar por teléfono. Esto no borra el registro.

In [ ]:
def valores_informados(serie):
    return sorted(set(str(v) for v in serie.dropna()))

filas_consolidadas = []
for identificador, grupo in consultas_resueltas.groupby('lead_consolidado_id', sort=True):
    nombres = valores_informados(grupo['nombre_cliente_normalizado'])
    nombre_presentacion = max(nombres, key=lambda v: (
        sum(len(t.strip('.')) > 1 for t in v.split()), len(v)
    )) if nombres else pd.NA
    filas_consolidadas.append({
        'lead_consolidado_id': identificador,
        'empresa_id': grupo['empresa_id_normalizado'].iloc[0],
        'nombre_presentacion': nombre_presentacion,
        'nombres_declarados': nombres,
        'telefonos': valores_informados(grupo['telefono_normalizado']),
        'telefono_utilizable': bool(grupo['telefono_estado'].eq('valido').all()),
        'correos': valores_informados(grupo['email_normalizado']),
        'varios_correos': grupo['email_normalizado'].nunique(dropna=True) > 1,
        'ciudades': valores_informados(grupo['ciudad_normalizado']),
        'canales': valores_informados(grupo['canal_normalizado']),
        'puntos_venta': valores_informados(grupo['punto_venta_id_normalizado']),
        'modelos_identificados': valores_informados(grupo['modelo_interes_texto_normalizado']),
        'modelos_declarados': valores_informados(grupo['modelo_interes_texto']),
        'consultas_con_modelo_pendiente': int((~grupo['modelo_estado'].isin(['coincidencia_exacta', 'abreviatura_unica'])).sum()),
        'primera_fecha_registro_resuelta': grupo['fecha_registro_normalizado'].min(),
        'ultima_fecha_registro_resuelta': grupo['fecha_registro_normalizado'].max(),
        'consultas_con_fecha_registro_pendiente': int(grupo['fecha_registro_normalizado'].isna().sum()),
        'lead_ids_origen': grupo['lead_id_normalizado'].tolist(),
        'numero_consultas': len(grupo),
        'reglas_identidad': valores_informados(grupo['regla_identidad']),
    })
leads_consolidados = pd.DataFrame(filas_consolidadas)
mapa_origen_consolidado = trazabilidad_filas.merge(
    consultas_resueltas[['fila_origen', 'lead_consolidado_id', 'regla_identidad']].rename(columns={'fila_origen': 'fila_conservada'}),
    on='fila_conservada', how='left', validate='many_to_one',
)

# Enlazar conversaciones sin cambiar sus mensajes ni inventar vínculos ausentes.
referencia_identidad = consultas_resueltas[['lead_id_normalizado', 'empresa_id_normalizado', 'lead_consolidado_id']]
conversaciones_resueltas = datos_normalizados['conversaciones'].merge(
    referencia_identidad, on='lead_id_normalizado', how='left', validate='many_to_one', indicator=True,
)
print(f'Filas recibidas: {len(leads)}')
print(f'Consultas distintas conservadas: {len(consultas_resueltas)}')
print(f'Identidades consolidadas dentro de cada empresa: {len(leads_consolidados)}')
print('Grupos resueltos con consultas de varios canales:')
multicanal = leads_consolidados.loc[leads_consolidados['canales'].map(len).gt(1)]
display(multicanal[['lead_consolidado_id', 'empresa_id', 'nombre_presentacion', 'canales', 'lead_ids_origen']])
print('Primeras 10 identidades consolidadas:')
with pd.option_context('display.max_columns', None):
    display(leads_consolidados.head(10))

## 16. Excepciones de normalización y comprobación del alcance

Normalizar no significa inventar datos. Los valores que no se pueden determinar con la evidencia disponible se conservan como originales y quedan en `excepciones_normalizacion`. Un estado `faltante` se distingue de `invalida`, `ambigua` o `no_reconocido`. Las fechas salen en ISO y su orden de entrada se corrige por lead; las abreviaturas únicas de modelos mantienen su marca de inferencia.

El orden de entrada se evalúa por lead con las reglas de la sección 8; la salida siempre usa año-mes-día. No se asigna un SKU a una marca sin modelo. Esto permite que la ejecución termine sin intervención manual, conservando un reporte explícito de lo que requeriría mejor información de origen.

La sección no implementa la extracción de IA de WhatsApp, scoring ni base de datos: son etapas posteriores del assessment. El notebook se ejecuta completo desde un entorno Python con pandas e IPython; no hay que editar registros o ejecutar decisiones manuales entre celdas.

In [ ]:
filas_excepciones = []
campos_originales = {
    'leads': list(leads.columns), 'asesores': list(asesores.columns),
    'catalogo_motos': list(catalogo_motos.columns), 'historico_cierres': list(historico_cierres.columns),
    'conversaciones': list(conversaciones_df.columns),
}
estados_resueltos = {'valido', 'valida', 'resuelta_por_contexto', 'coincidencia_exacta', 'abreviatura_unica'}
for nombre, tabla in datos_normalizados.items():
    columna_id = {'leads': 'lead_id', 'historico_cierres': 'lead_id', 'asesores': 'asesor_id',
                  'catalogo_motos': 'sku', 'conversaciones': 'conversacion_id'}[nombre]
    for columna in [c for c in tabla.columns if c.endswith('_estado')]:
        campo = columna[:-len('_estado')]
        if campo == 'modelo':
            campo = 'modelo_interes_texto' if nombre == 'leads' else 'modelo_cotizado'
        if campo not in campos_originales[nombre]:
            continue
        for indice, fila in tabla.loc[~tabla[columna].isin(estados_resueltos)].iterrows():
            filas_excepciones.append({'tabla': nombre, 'fila_origen': indice, 'id': fila[columna_id],
                'campo': campo, 'valor_original': fila[campo], 'estado': fila[columna],
                'comentario': fila.get(campo + '_criterio', 'Revisar el estado indicado y el valor original.')})
excepciones_normalizacion = pd.DataFrame(filas_excepciones)
print('Excepciones por tabla, campo y estado (los faltantes no equivalen a errores):')
with pd.option_context('display.max_rows', None):
    display(excepciones_normalizacion.groupby(['tabla', 'campo', 'estado']).size().to_frame('valores'))
print('Grupos que no se pudieron resolver por identidad:')
identidades_pendientes = decisiones_identidad.loc[decisiones_identidad['decision'].ne('agrupado')]
display(identidades_pendientes)

# Comprobaciones ejecutables de integridad del resultado.
assert len(mapa_origen_consolidado) == len(leads), 'Se perdió trazabilidad de filas.'
assert mapa_origen_consolidado['lead_consolidado_id'].notna().all()
assert leads_consolidados['lead_consolidado_id'].is_unique
assert consultas_resueltas.groupby('lead_consolidado_id')['empresa_id_normalizado'].nunique().eq(1).all(), 'Se mezclaron empresas.'
assert set(consultas_resueltas['lead_id_normalizado']) == set(datos_normalizados['leads']['lead_id_normalizado'])
assert leads_consolidados['numero_consultas'].sum() == len(consultas_resueltas)
assert len(conversaciones_resueltas) == len(conversaciones)
assert conversaciones_resueltas.loc[conversaciones_resueltas['_merge'].eq('left_only'), 'lead_consolidado_id'].isna().all()
print('Comprobaciones superadas: trazabilidad completa, consultas conservadas y separación por empresa.')

## 17. Teléfono inválido: conservar valor útil o excluir del resultado limpio

La revisión del lead `LD-01501` muestra teléfono `300123`, nombre `prueba prueba`, fecha imposible `2026-08-33`, sin correo, ciudad, canal, modelo, campaña ni conversaciones. Su estado es `Sin gestión`. No aporta información comercial recuperable: se excluye de las salidas limpias, como se solicitó.

La regla es deliberadamente conservadora: solo excluir automáticamente un registro con teléfono inválido/faltante cuando el nombre está vacío o es un marcador de prueba explícito y no hay fecha de registro válida, correo, modelo, ciudad, canal, campaña, contacto, gestión comercial útil ni conversación. Un teléfono inválido por sí solo nunca basta para excluir.

Si hay alguna señal útil, se conserva el registro con `conservar_con_anotacion` y el motivo. La exclusión se aplica a `consultas_limpias`, `leads_limpios` y `conversaciones_limpias`; las tablas de diagnóstico y archivos fuente siguen intactos para auditoría. `trazabilidad_salida_limpia` permite identificar todas las filas excluidas.

In [ ]:
ids_con_conversacion = set(datos_normalizados['conversaciones']['lead_id_normalizado'].dropna())
marcadores_prueba = {'prueba', 'prueba prueba', 'test', 'test test'}
revision_telefonos_invalidos = []
for indice, fila in consultas_resueltas.loc[consultas_resueltas['telefono_estado'].ne('valido')].iterrows():
    utiles = []
    nombre = clave_texto(fila['nombre_cliente'])
    if nombre and nombre not in marcadores_prueba:
        utiles.append('nombre distinto de un marcador de prueba')
    for campo in ['email_normalizado', 'modelo_interes_texto', 'ciudad_normalizado', 'canal_normalizado', 'campania_normalizado']:
        if pd.notna(limpiar_texto(fila[campo])):
            utiles.append(campo)
    if pd.notna(fila['fecha_registro_normalizado']):
        utiles.append('fecha de registro válida')
    if pd.notna(fila['fecha_primer_contacto_normalizado']):
        utiles.append('primer contacto válido')
    if pd.notna(fila['estado_gestion_normalizado']) and fila['estado_gestion_normalizado'] != 'Sin gestión':
        utiles.append('gestión comercial registrada')
    if fila['lead_id_normalizado'] in ids_con_conversacion:
        utiles.append('conversación asociada')
    decision = 'conservar_con_anotacion' if utiles else 'excluir_sin_valor_recuperable'
    comentario = ('Teléfono inválido; conservar porque contiene: ' + ', '.join(utiles) + '.' if utiles
                  else 'Teléfono inválido, nombre vacío/de prueba y sin señales comerciales recuperables ni conversación. Excluido solo del resultado limpio.')
    revision_telefonos_invalidos.append({'fila_origen': indice, 'lead_id': fila['lead_id_normalizado'],
        'empresa': fila['empresa_id_normalizado'], 'telefono_original': fila['telefono'],
        'telefono_estado': fila['telefono_estado'], 'decision': decision, 'comentario': comentario})
revision_telefonos_invalidos = pd.DataFrame(revision_telefonos_invalidos)
filas_excluidas = set(revision_telefonos_invalidos.loc[
    revision_telefonos_invalidos['decision'].eq('excluir_sin_valor_recuperable'), 'fila_origen'])
registros_excluidos = consultas_resueltas.loc[consultas_resueltas.index.isin(filas_excluidas)].copy()
consultas_limpias = consultas_resueltas.loc[~consultas_resueltas.index.isin(filas_excluidas)].copy()
consultas_limpias['anotacion_telefono'] = consultas_limpias.index.map(
    revision_telefonos_invalidos.set_index('fila_origen')['comentario'].to_dict())
ids_limpios = set(consultas_limpias['lead_consolidado_id'])
# La regla de exclusión solo alcanza teléfonos inválidos, que nunca se agruparon con otros leads.
leads_limpios = leads_consolidados.loc[leads_consolidados['lead_consolidado_id'].isin(ids_limpios)].copy()
# Conservar TODAS las conversaciones, incluso las que no pueden vincularse.
conversaciones_limpias = conversaciones_resueltas.copy(deep=True)
vinculo_valido = conversaciones_limpias['lead_consolidado_id'].isin(ids_limpios)
conversaciones_limpias['lead_id_recibido'] = conversaciones_limpias['lead_id']
conversaciones_limpias['lead_id_vinculado'] = conversaciones_limpias['lead_id_normalizado'].where(vinculo_valido, pd.NA)
conversaciones_limpias.loc[~vinculo_valido, ['lead_consolidado_id', 'empresa_id_normalizado']] = pd.NA
conversaciones_limpias['estado_vinculo'] = vinculo_valido.map({True: 'identificado', False: 'lead_y_empresa_desconocidos'})
conversaciones_limpias['anotacion_vinculo'] = vinculo_valido.map({
    True: 'Vinculada al lead de origen y a su empresa.',
    False: 'El ID recibido no corresponde a un lead conservado. Se mantiene la conversación íntegra; lead y empresa vinculados desconocidos (NULL para base de datos).',
})
# Es una vista de pendientes que también están incluidos en conversaciones_limpias.
conversaciones_pendientes = conversaciones_limpias.loc[~vinculo_valido].copy()
trazabilidad_salida_limpia = mapa_origen_consolidado.copy()
trazabilidad_salida_limpia['estado_salida'] = trazabilidad_salida_limpia['fila_conservada'].map(
    lambda f: 'excluido_sin_valor_recuperable' if f in filas_excluidas else 'conservado')
trazabilidad_salida_limpia['motivo_exclusion'] = trazabilidad_salida_limpia['fila_conservada'].map(
    revision_telefonos_invalidos.loc[revision_telefonos_invalidos['decision'].eq('excluir_sin_valor_recuperable')].set_index('fila_origen')['comentario'].to_dict())
fechas_por_revisar['estado_registro'] = fechas_por_revisar.apply(
    lambda r: 'excluido_sin_valor_recuperable' if r['tabla'] == 'leads' and r['fila_dataframe'] in filas_excluidas else 'conservado', axis=1)

with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
    display(revision_telefonos_invalidos)
print(f'Consultas en salida limpia: {len(consultas_limpias)}')
print(f'Identidades en salida limpia: {len(leads_limpios)}')
print(f'Registros excluidos con trazabilidad: {len(registros_excluidos)}')
assert len(consultas_limpias) + len(registros_excluidos) == len(consultas_resueltas)
assert consultas_limpias['lead_consolidado_id'].isin(leads_limpios['lead_consolidado_id']).all()
assert leads_limpios['numero_consultas'].sum() == len(consultas_limpias)
assert len(conversaciones_limpias) == len(conversaciones_resueltas)
assert conversaciones_pendientes['lead_id_vinculado'].isna().all()
assert conversaciones_pendientes['empresa_id_normalizado'].isna().all()

## 18. Decisiones de conservación y registro de incidencias

- **Modelos:** conservar todos los intereses declarados, incluidos marcas solas, modelos ambiguos y campos faltantes. Un SKU desconocido permanece como NULL; no se elimina el lead por no identificar su modelo. La exclusión de `prueba prueba` responde a ausencia de valor recuperable, no a su modelo.
- **Catálogo y mensajes:** conservar los datos originales. Las verificaciones de disponibilidad son observaciones, no correcciones del catálogo.
- **Conversaciones:** conservar las 677 conversaciones y todos sus mensajes. Cuando el lead no existe, preservar `lead_id_recibido` para rastrear la fuente, dejar `lead_id_vinculado`, `lead_consolidado_id` y empresa en NULL y marcar `lead_y_empresa_desconocidos`. No se crea un cliente ficticio compartido ni se atribuye una empresa.
- **Exclusiones:** documentar cada fila exacta repetida y cada registro descartado, con su motivo y vínculo al origen. El archivo fuente no se modifica.

### JSON de auditoría

Al ejecutar el notebook, generar `outputs/calidad_datos/incidencias_datos.json`. Incluye faltantes (que no necesariamente son errores), valores inválidos, correcciones de formato, reglas inferidas, duplicados detectados/resueltos, referencias faltantes, observaciones comerciales y exclusiones. Cada evento tiene origen, ID, campo cuando aplica, valor original, resultado, acción y motivo. Los conteos son de eventos, no de personas; una fila puede tener varias incidencias.

Las correcciones de formato se registran para demostrar lo detectado y transformado. Las coincidencias de teléfono entre empresas se registran como observación: se mantienen separadas. Este JSON es un registro de calidad de los datos sintéticos del ejercicio, no el almacenamiento final de la solución.

In [ ]:
from collections import Counter

def valor_json(valor):
    if isinstance(valor, dict):
        return {str(k): valor_json(v) for k, v in valor.items()}
    if isinstance(valor, (list, tuple, set)):
        return [valor_json(v) for v in valor]
    if valor is None or valor is pd.NA or valor is pd.NaT:
        return None
    if isinstance(valor, pd.Timestamp):
        return valor.isoformat(sep=' ')
    if hasattr(valor, 'item'):
        return valor_json(valor.item())
    if isinstance(valor, float) and not math.isfinite(valor):
        return None
    return valor

incidencias = []

def registrar(tipo, tabla, fila, identificador, campo, original, resultado, accion, motivo, detalle=None):
    incidencias.append(valor_json({
        'incidencia_id': f'INC-{len(incidencias) + 1:06d}', 'tipo': tipo,
        'tabla': tabla, 'fila_dataframe': fila, 'id_origen': identificador,
        'campo': campo, 'valor_original': original, 'valor_resultante': resultado,
        'accion': accion, 'motivo': motivo, 'detalle': detalle,
    }))

id_tabla = {'leads': 'lead_id', 'asesores': 'asesor_id', 'catalogo_motos': 'sku',
            'historico_cierres': 'lead_id', 'conversaciones': 'conversacion_id'}
# Revisar cada campo original, incluidos vacíos que no tenían validación específica.
for nombre, tabla in datos_normalizados.items():
    for indice, fila in tabla.iterrows():
        identificador = fila[id_tabla[nombre]]
        for campo in campos_originales[nombre]:
            if campo == 'mensajes':
                continue
            original = fila[campo]
            normalizado = fila.get(campo + '_normalizado', original)
            estado = fila.get(campo + '_estado')
            if campo in ['modelo_interes_texto', 'modelo_cotizado']:
                estado = fila.get('modelo_estado')
            comentario = fila.get(campo + '_criterio')
            vacio = pd.isna(original) or (isinstance(original, str) and not original.strip())
            if vacio:
                registrar('faltante', nombre, indice, identificador, campo, original, normalizado,
                          'conservar_sin_imputar', comentario or 'Campo no informado; no se inventa un valor. Puede ser opcional según el contexto.')
            elif estado and estado not in estados_resueltos:
                registrar('valor_no_resuelto', nombre, indice, identificador, campo, original, normalizado,
                          'conservar_original_y_marcar', comentario or f'Estado de validación: {estado}. El valor no se reemplaza por una suposición.',
                          {'estado': estado, 'candidatos': fila.get('modelo_candidatos') if campo.startswith('modelo') else None})
            elif campo + '_normalizado' in tabla.columns and valor_json(original) != valor_json(normalizado):
                registrar('normalizacion', nombre, indice, identificador, campo, original, normalizado,
                          'normalizar_en_copia', comentario or 'Aplicación de las reglas explícitas de limpieza, equivalencia o conversión de tipo del notebook.',
                          {'estado': estado})
            if estado in ['abreviatura_unica', 'resuelta_por_contexto']:
                registrar('inferencia_documentada', nombre, indice, identificador, campo, original, normalizado,
                          'conservar_marca_de_inferencia', comentario or 'Asignación a la única referencia compatible del catálogo, según la regla de abreviaturas.')


for indice, fila in datos_normalizados['leads'].loc[datos_normalizados['leads']['orden_fechas_detectado'].isin([
    'MDY_por_evidencia_del_lead', 'origen_mixto_resuelto_por_valores', 'MDY_por_coherencia_del_lead', 'origen_mixto_por_coherencia_unica'])].iterrows():
    registrar('orden_fechas_corregido', 'leads', indice, fila['lead_id'], 'fechas_del_lead',
              {'registro': fila['fecha_registro'], 'contacto': fila['fecha_primer_contacto']},
              {'registro': fila['fecha_registro_normalizado'], 'contacto': fila['fecha_primer_contacto_normalizado']},
              fila['orden_fechas_detectado'], fila['comentario_orden_fechas'])

for _, fila in fechas_por_revisar.loc[fechas_por_revisar['estado'].eq('secuencia_inconsistente')].iterrows():
    registrar('secuencia_fechas', fila['tabla'], fila['fila_dataframe'], fila['id'], fila['campo'], fila['original'], fila['normalizado'],
              'conservar_y_marcar', fila['criterio'])
for _, fila in trazabilidad_filas.loc[trazabilidad_filas['accion'].eq('repeticion_exacta')].iterrows():
    registrar('duplicado_exacto', 'leads', fila['fila_origen'], fila['lead_id_origen'], None,
              leads.loc[fila['fila_origen']].to_dict(), {'fila_conservada': fila['fila_conservada']},
              'excluir_repeticion_de_salida', 'Todas las columnas originales son idénticas a otra fila; se conserva la primera aparición y el vínculo de trazabilidad.')
for _, fila in decisiones_identidad.iterrows():
    registrar('duplicado_identidad', 'leads', None, fila['lead_ids'], None, fila['nombres_originales'], fila['decision'],
              'agrupar_sin_perder_consultas' if fila['decision'] == 'agrupado' else 'mantener_separados',
              'Misma empresa, teléfono válido y evaluación explícita de compatibilidad de todos los pares de nombres.', fila.to_dict())
for telefono, grupo in datos_normalizados['leads'].loc[datos_normalizados['leads']['telefono_estado'].eq('valido')].groupby('telefono_normalizado'):
    if grupo['empresa_id_normalizado'].nunique() > 1:
        registrar('telefono_en_varias_empresas', 'leads', None, grupo['lead_id'].tolist(), 'telefono', telefono, telefono,
                  'mantener_separados_por_empresa', 'Coincidencia entre empresas; nunca se fusionan identidades de empresas distintas.',
                  {'empresas': sorted(grupo['empresa_id_normalizado'].unique().tolist())})
for _, fila in conversaciones_pendientes.iterrows():
    registrar('conversacion_sin_lead', 'conversaciones', None, fila['conversacion_id'], 'lead_id', fila['lead_id_recibido'], None,
              'conservar_con_vinculo_desconocido', fila['anotacion_vinculo'], {'empresa': None, 'estado_vinculo': fila['estado_vinculo']})
for _, fila in revision_telefonos_invalidos.iterrows():
    registrar('exclusion_registro' if fila['decision'].startswith('excluir') else 'telefono_invalido_con_valor',
              'leads', fila['fila_origen'], fila['lead_id'], 'telefono', fila['telefono_original'], None,
              fila['decision'], fila['comentario'], leads.loc[fila['fila_origen']].to_dict())
for _, fila in leads_modelo_no_disponible.iterrows():
    registrar('modelo_no_listado_en_punto', 'leads', fila['fila_dataframe'], fila['lead_id_normalizado'], 'modelo_interes_texto',
              fila['modelo_interes_texto'], fila['modelo_sku'], 'conservar_interes_y_catalogo',
              'El SKU identificado no está listado en el punto asignado. Observación comercial, no error demostrado del catálogo.',
              {'punto': fila['punto_venta_id_normalizado']})
for _, fila in leads_punto_inconsistente.iterrows():
    registrar('empresa_punto_no_reconocido', 'leads', fila['fila_dataframe'], fila['lead_id_normalizado'], 'punto_venta_id',
              fila['punto_venta_id_normalizado'], None, 'conservar_y_marcar', 'El par empresa/punto no aparece en la referencia de asesores.')

# Verificar los mensajes sin modificar su texto ni orden.
for conversacion in conversaciones:
    for posicion, mensaje in enumerate(conversacion['mensajes'], 1):
        for campo in ['emisor', 'hora', 'texto']:
            if not mensaje.get(campo):
                registrar('mensaje_campo_faltante', 'conversaciones', None, conversacion['conversacion_id'], campo,
                          mensaje.get(campo), None, 'conservar_mensaje', 'Campo ausente en mensaje original.', {'orden_mensaje': posicion})

reporte_incidencias = {
    'version': 1,
    'fuente': 'notebooks/01_importacion_datos.ipynb y data/raw',
    'politicas': {
        'separacion_empresa': 'No fusionar entre empresas, aunque coincidan nombre, teléfono y correo.',
        'modelos': 'Conservar todos los intereses, sin excluir leads por modelo faltante o ambiguo.',
        'conversaciones': 'Conservar todas; vínculo desconocido como null y marca explícita, preservando el ID recibido.',
        'eliminaciones': 'Solo en salidas limpias; conservar fuente y explicación de cada exclusión.',
        'fechas': 'Consultar la regla y los comentarios de la sección 8; cada corrección figura por campo.',
    },
    'resumen': {
        'eventos': len(incidencias), 'por_tipo': dict(Counter(i['tipo'] for i in incidencias)),
        'filas_leads_recibidas': len(leads), 'consultas_conservadas': len(consultas_limpias),
        'identidades_conservadas': len(leads_limpios), 'conversaciones_conservadas': len(conversaciones_limpias),
        'conversaciones_sin_vinculo': len(conversaciones_pendientes), 'registros_excluidos_sin_valor': len(registros_excluidos),
    },
    'incidencias': incidencias,
}
ruta_incidencias = raiz_proyecto / 'outputs' / 'calidad_datos' / 'incidencias_datos.json'
ruta_incidencias.parent.mkdir(parents=True, exist_ok=True)
ruta_incidencias.write_text(json.dumps(reporte_incidencias, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
print(f'JSON generado: {ruta_incidencias}')
display(pd.DataFrame(reporte_incidencias['resumen']['por_tipo'].items(), columns=['tipo', 'eventos']))
assert len(conversaciones_limpias) == len(conversaciones)
assert conversaciones_limpias['mensajes'].tolist() == [c['mensajes'] for c in conversaciones]
assert consultas_limpias['modelo_interes_texto'].equals(consultas_resueltas.loc[consultas_limpias.index, 'modelo_interes_texto'])

## 19. Exportación reproducible a `data/processed`

El notebook completo genera los archivos sin editar datos a mano. `LD-01501` (`prueba prueba`) queda fuera de leads y consultas. Se conserva únicamente como evidencia en los archivos fuente y de auditoría.

| Archivo | Contenido |
| --- | --- |
| `leads.csv` | Una identidad por empresa, clave `lead_consolidado_id`. Correos, intereses, canales e IDs de origen se conservan como listas JSON dentro del CSV. |
| `consultas_leads.csv` | Una fila por consulta distinta, clave `lead_id`, vinculada por `lead_consolidado_id`. Campos normalizados, originales en columnas `_original` y estados de validación. |
| `conversaciones.json` | Todas las conversaciones y mensajes. `lead_id` apunta a la consulta; `lead_consolidado_id` a la identidad. Si se desconocen, ambos y `empresa_id` son null; `lead_id_recibido` conserva la referencia fuente. |
| `asesores.csv` | Asesores normalizados, originales y validaciones. |
| `catalogo_motos.csv` | Copia exacta del catálogo recibido. |
| `historico_cierres.csv` | Histórico normalizado, originales y estados. Sus IDs no pertenecen al conjunto actual. |
| `trazabilidad_leads.csv` | Todas las filas recibidas, sus representantes, consolidaciones y motivos de exclusión. No es una tabla de clientes activos. |
| `incidencias_datos.json` | Hallazgos, correcciones y exclusiones con explicación. |
| `manifest.json` | Conteos y hashes SHA-256 de fuentes y resultados. |
| `README.md` | Guía de uso de estos archivos. |

CSV: UTF-8, separador coma, faltantes vacíos. Listas como JSON. JSON: desconocidos como `null`, nunca `NaN`. Fechas en año-mes-día, con hora solo si la fuente la informa; la precisión se conserva. Se vuelven a generar los mismos nombres al ejecutar el notebook.

Estos archivos completan la etapa de preparación y no reemplazan la base de datos final exigida por la prueba.

In [ ]:
import shutil

ruta_processed = raiz_proyecto / 'data' / 'processed'
ruta_processed.mkdir(parents=True, exist_ok=True)

def fecha_exportable(valor, precision=None):
    if pd.isna(valor):
        return None
    return pd.Timestamp(valor).strftime('%Y-%m-%d' if precision == 'fecha' else '%Y-%m-%d %H:%M:%S')

def tabla_exportable(tabla, columnas_fuente):
    salida = pd.DataFrame(index=tabla.index)
    for campo in columnas_fuente:
        normalizado = campo + '_normalizado'
        if normalizado in tabla.columns:
            salida[campo + '_original'] = tabla[campo]
            if pd.api.types.is_datetime64_any_dtype(tabla[normalizado]):
                salida[campo] = [fecha_exportable(v, precision) for v, precision in zip(
                    tabla[normalizado], tabla.get(campo + '_precision', pd.Series(None, index=tabla.index)))]
            else:
                salida[campo] = tabla[normalizado]
        else:
            salida[campo] = tabla[campo]
    for campo in tabla.columns:
        if campo not in columnas_fuente and not campo.endswith('_normalizado'):
            salida[campo] = tabla[campo]
    return salida

def escribir_csv(tabla, ruta):
    salida = tabla.copy(deep=True)
    for campo in salida.columns:
        if pd.api.types.is_datetime64_any_dtype(salida[campo]):
            salida[campo] = salida[campo].map(fecha_exportable)
        salida[campo] = salida[campo].map(lambda v: json.dumps(valor_json(v), ensure_ascii=False, allow_nan=False)
                                        if isinstance(v, (list, tuple, dict, set)) else v)
    salida.to_csv(ruta, index=False, encoding='utf-8', lineterminator='\n', na_rep='')

consultas_exportadas = tabla_exportable(consultas_limpias, list(leads.columns))
asesores_exportados = tabla_exportable(datos_normalizados['asesores'], list(asesores.columns))
historico_exportado = tabla_exportable(datos_normalizados['historico_cierres'], list(historico_cierres.columns))
exportaciones_csv = {
    'leads.csv': leads_limpios, 'consultas_leads.csv': consultas_exportadas,
    'asesores.csv': asesores_exportados, 'historico_cierres.csv': historico_exportado,
    'trazabilidad_leads.csv': trazabilidad_salida_limpia,
}
for nombre, tabla in exportaciones_csv.items():
    escribir_csv(tabla, ruta_processed / nombre)
shutil.copyfile(ruta_datos / 'catalogo_motos.csv', ruta_processed / 'catalogo_motos.csv')

conversaciones_exportadas = []
for _, fila in conversaciones_limpias.iterrows():
    conversaciones_exportadas.append(valor_json({
        'conversacion_id': fila['conversacion_id_normalizado'],
        'lead_id': fila['lead_id_vinculado'], 'lead_consolidado_id': fila['lead_consolidado_id'],
        'empresa_id': fila['empresa_id_normalizado'], 'lead_id_recibido': fila['lead_id_recibido'],
        'canal': fila['canal_normalizado'],
        'fecha_inicio': fecha_exportable(fila['fecha_inicio_normalizado'], fila['fecha_inicio_precision']),
        'fecha_inicio_original': fila['fecha_inicio'], 'fecha_inicio_estado': fila['fecha_inicio_estado'],
        'fecha_inicio_criterio': fila['fecha_inicio_criterio'],
        'estado_vinculo': fila['estado_vinculo'], 'anotacion_vinculo': fila['anotacion_vinculo'],
        'mensajes': fila['mensajes'],
    }))
(ruta_processed / 'conversaciones.json').write_text(
    json.dumps(conversaciones_exportadas, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
shutil.copyfile(ruta_incidencias, ruta_processed / 'incidencias_datos.json')
conteos_exportados = {nombre: len(tabla) for nombre, tabla in exportaciones_csv.items()}
conteos_exportados.update({'catalogo_motos.csv': len(catalogo_motos), 'conversaciones.json': len(conversaciones_exportadas),
                          'incidencias_datos.json': len(reporte_incidencias['incidencias'])})
manifest = {
    'version': 1, 'generador': 'notebooks/01_importacion_datos.ipynb',
    'codificacion': 'UTF-8', 'separador_csv': ',',
    'nota_conteos': 'Incidencias cuenta eventos; trazabilidad incluye filas excluidas. No son clientes activos.',
    'fuentes': {archivo.name: hashlib.sha256(archivo.read_bytes()).hexdigest()
                for archivo in sorted(ruta_datos.iterdir()) if archivo.suffix in ['.csv', '.json']},
    'archivos': {nombre: {'registros': cantidad, 'sha256': hashlib.sha256((ruta_processed / nombre).read_bytes()).hexdigest()}
                 for nombre, cantidad in conteos_exportados.items()},
    'politicas': reporte_incidencias['politicas'],
}
(ruta_processed / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
(ruta_processed / 'README.md').write_text("""# Datos procesados

Generados ejecutando completo `notebooks/01_importacion_datos.ipynb`. No editar a mano: se regeneran.

## Archivos principales

- `leads.csv`: una identidad por empresa, clave `lead_consolidado_id`.
- `consultas_leads.csv`: consultas distintas con `lead_id` de origen y vínculo `lead_consolidado_id`. Incluye valores normalizados, originales y diagnósticos.
- `conversaciones.json`: todas las conversaciones y mensajes. `lead_id` enlaza a consultas y `lead_consolidado_id` a leads. Los vínculos desconocidos son null; `lead_id_recibido` conserva la referencia original.
- `asesores.csv`: asesores normalizados, con originales y validaciones.
- `catalogo_motos.csv`: copia exacta del original.
- `historico_cierres.csv`: histórico normalizado; sus leads no pertenecen al conjunto actual.

## Auditoría

- `trazabilidad_leads.csv`: todas las filas recibidas, incluidas repeticiones y el registro excluido, con motivos.
- `incidencias_datos.json`: hallazgos y decisiones. Incluye el registro excluido como evidencia; no es un cliente activo.
- `manifest.json`: conteos y hashes de archivos fuente y generados.

## Convenciones y límites

CSV en UTF-8, coma como separador y faltantes vacíos. Las listas se guardan como JSON dentro del CSV. JSON usa null para desconocidos. Fechas normalizadas en año-mes-día, con hora cuando existe; las columnas `_original` mantienen el valor recibido.

Los modelos ambiguos o ausentes se conservan. LD-01501 (prueba prueba) se excluye por no aportar valor recuperable. Las conversaciones sin lead siguen presentes con empresa desconocida. Las identidades nunca se fusionan entre empresas.

La identidad y algunas interpretaciones de fecha se resuelven con reglas documentadas: son inferencias, no verificación de identidad real. Fechas vacías permanecen vacías. Las fechas agregadas de leads no deben tratarse como horas exactas cuando la fuente solo informa el día.

Estos archivos completan ingestión, normalización y consolidación. No sustituyen la base de datos final exigida por la prueba.
""", encoding='utf-8')
print(f'Archivos exportados a: {ruta_processed}')
display(pd.DataFrame(conteos_exportados.items(), columns=['archivo', 'registros']))

### Verificación de los archivos guardados

Reabrimos los archivos para verificar conteos, vínculos, mensajes y separación por empresa. Confirmamos que `LD-01501` no es un lead ni una consulta activa; permanece únicamente en la auditoría y trazabilidad. Estas comprobaciones forman parte de la ejecución del notebook.

In [ ]:
leads_guardados = pd.read_csv(ruta_processed / 'leads.csv', dtype={'lead_consolidado_id': 'string', 'empresa_id': 'string'})
consultas_guardadas = pd.read_csv(ruta_processed / 'consultas_leads.csv', dtype={'lead_id': 'string', 'telefono': 'string'})
conversaciones_guardadas = json.loads((ruta_processed / 'conversaciones.json').read_text(encoding='utf-8'))
assert len(leads_guardados) == len(leads_limpios)
assert len(consultas_guardadas) == len(consultas_limpias)
assert len(conversaciones_guardadas) == len(conversaciones)
assert 'LD-01501' not in set(consultas_guardadas['lead_id'])
assert all('LD-01501' not in json.loads(ids) for ids in leads_guardados['lead_ids_origen'])
assert leads_guardados['lead_consolidado_id'].is_unique
assert consultas_guardadas['lead_id'].is_unique
empresas_salida = leads_guardados.set_index('lead_consolidado_id')['empresa_id'].to_dict()
assert all(empresas_salida[fila['lead_consolidado_id']] == fila['empresa_id'] for _, fila in consultas_guardadas.iterrows())
consulta_por_id = consultas_guardadas.set_index('lead_id').to_dict('index')
for original, guardada in zip(conversaciones, conversaciones_guardadas):
    assert original['conversacion_id'] == guardada['conversacion_id']
    assert original['lead_id'] == guardada['lead_id_recibido']
    assert original['mensajes'] == guardada['mensajes']
    if guardada['lead_id'] is None:
        assert guardada['empresa_id'] is None and guardada['lead_consolidado_id'] is None
        assert guardada['estado_vinculo'] == 'lead_y_empresa_desconocidos'
    else:
        consulta = consulta_por_id[guardada['lead_id']]
        assert guardada['lead_consolidado_id'] == consulta['lead_consolidado_id']
        assert guardada['empresa_id'] == consulta['empresa_id']
assert (ruta_processed / 'catalogo_motos.csv').read_bytes() == (ruta_datos / 'catalogo_motos.csv').read_bytes()
for nombre in ['asesores.csv', 'catalogo_motos.csv', 'historico_cierres.csv', 'trazabilidad_leads.csv']:
    assert len(pd.read_csv(ruta_processed / nombre)) == conteos_exportados[nombre]
for nombre, detalle in manifest['archivos'].items():
    assert hashlib.sha256((ruta_processed / nombre).read_bytes()).hexdigest() == detalle['sha256']
print('Exportación verificada. LD-01501 excluido; conversaciones y consultas útiles conservadas; empresas separadas.')
print('Alcance completado: ingesta de los cinco archivos, normalización y resolución de duplicados con reglas y excepciones documentadas.')